# EvalDesk v8 — 3-input, parts-aware grading (Chandra-2 OCR · equation+diagram HTML · Qwen3-8B + STS/NLI flagging)

**Built on Krishnarsh & Nancy's v6 notebook + v7.5 portal + the equation/diagram HTML pipeline.**

Three inputs, each handled separately:
1. **Question paper** → per-question **and per-part** marks are extracted from the `[N]` brackets (no auto-rubric for the *totals*).
2. **Correct answer key** → OCR'd → split per question→part → **downloadable structured HTML** (text, equations via MathJax, diagrams cropped & embedded).
3. **Student answer sheet** → leading instruction page(s) skipped, **blank pages skipped by handwriting-ink** (barcode band + printed rules stripped first), OCR'd → **downloadable HTML** → mapped to the key's parts → graded.

Grading = **Qwen3-8B** (4-bit) leads; the **STS+NLI engine cross-checks and flags** disagreements for review. VRAM: grading models park on CPU during OCR, **Chandra is unloaded after OCR**, then graders + the 8B load — so a single L4/24 GB never OOMs.

> **NEW seams to validate on a GPU** (everything else is lifted verbatim from your working files): (a) `extract_qp_marks` on the question paper, (b) `split_into_parts` on OCR'd handwriting, (c) parts-aware key build (Part 2.3), (d) parts-aware mapping (Part 3.2), (e) `is_blank_page` thresholds (run `calibrate_blank_threshold` once), (f) the LLM/unload/flagging wiring. The OCR, preprocess, Tier-1, Tier-2, encoders and structured-HTML code is your own, unchanged.


## PART 1 — Setup (run once)
### 1.1 Install dependencies

In [ ]:
# OCR + evaluation dependencies. (openpyxl dropped — no XLSX export any more.)
# hf_transfer: ~2-3x faster download of the 10.6 GB Chandra checkpoint.
!pip install -q "chandra-ocr[hf]" pymupdf sentence-transformers transformers python-docx scipy pandas rapidfuzz markdown hf_transfer
# v8: + bitsandbytes/accelerate for the 4-bit Qwen3-8B grader
!pip install -q bitsandbytes accelerate
# v8.1: pymupdf (fitz) — read marks + question text from the question paper PDF text layer
!pip install -q pymupdf


### 1.2 Imports & configuration (OCR + LLM + blank-skip + 3 inputs)

In [ ]:
import os, gc, time
# Set before torch import so the allocator uses expandable segments,
# which drastically reduces fragmentation OOM errors on T4.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Faster HF Hub downloads for the 10.6 GB checkpoint (no-op if lib missing).
try:
    import hf_transfer  # noqa: F401
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
except ImportError:
    pass

import numpy as np
import cv2
import fitz  # PyMuPDF
from PIL import Image
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import rotate as nd_rotate

print("OpenCV version :", cv2.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("GPU            :", torch.cuda.get_device_name(0), f"(sm{cap[0]}{cap[1]})")


In [ ]:
# ── Prevent CUDA memory fragmentation (must be set BEFORE torch is imported) ──
# (os.environ is already set in the Imports cell; this is a safety fallback
#  in case you re-run this cell independently)
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ========================= EDIT THESE ===========================
# (No hardcoded file paths any more — you upload the answer key in Part 2
#  and each student sheet in Part 3.)
# ---- rendering ----
# 200 DPI is the sweet-spot for Chandra on a T4 (14 GiB).
# Drop to 150 if you still get OOM; raise to 250 only on an A100.
DPI = 200

# ---- Chandra ----
MODEL_ID    = "datalab-to/chandra-ocr-2"
PROMPT_TYPE = "ocr_layout"

# ---- OCR image cap ----
# Hard ceiling on the longest side of the image fed to Chandra.
# Chandra is a VLM — it sub-tiles the image internally, so very large
# images allocate huge attention tensors and cause OOM.
# 1400 px is safe on T4; raise to 1800 on A100, lower to 1024 on free-tier.
MAX_OCR_SIDE = 1400

# ---- Hough-transform deskew (OpenCV) ----
# Replaces the old MSER/PCA deskew, which over-rotated already-straight pages.
DESKEW_RANGE      = 10      # only trust skew estimates within +/- this many degrees
DESKEW_STEP       = 0.5     # projection-fallback angle granularity (degrees)
DESKEW_MIN_ANGLE  = 0.75    # dead-zone: |angle| below this is treated as straight (no rotation)
HOUGH_THRESHOLD   = 200     # Hough accumulator votes needed to accept a line
HOUGH_MIN_LEN_FR  = 0.30    # min line length as a fraction of page width
HOUGH_MAX_GAP     = 20      # max gap (px) to link collinear segments
HOUGH_MIN_LINES   = 8       # need >= this many near-horizontal lines to trust Hough
HOUGH_DILATE_W    = 25      # horizontal dilation width (px) to fuse glyphs into text lines

# ---- Adaptive Gaussian threshold ----
ADAPT_BLOCK = 31    # neighbourhood size (must be odd; forced below)
ADAPT_C     = 15    # constant subtracted from local mean

# ---- Denoise ----
DENOISE_MIN_AREA = 12   # remove ink blobs smaller than this (px); 0 = skip

# ---- v6 speed pack ----
OCR_DTYPE       = "auto"  # "auto" -> fp16 on T4-class GPUs (no native bf16 on sm<80), bf16 on A100+.
                          # Force with "fp16" / "bf16". fp16 is self-tested at load; falls back to bf16.
MAX_NEW_TOKENS  = 3072    # per-page output cap. chandra's default is 12,384 -> a single runaway
                          # page used to be able to burn 10+ minutes on a T4.
PAGES_PER_BATCH = 2       # pages per forward pass. Decode is memory-bandwidth-bound, so 2 pages
                          # ~= 1.7-1.9x throughput on T4. Auto-falls back to 1 on OOM.
LOOP_GUARD      = True    # abort a page early when the model gets stuck repeating itself.
RETRY_ON_LOOP   = True    # one extra sampled pass for a looped page (rare; recovers content
                          # after the loop point). Set False for pure speed.
# ================================================================

# ======================= v8 ADDITIONS =======================
# ---- Local LLM grader (Qwen3-8B, 4-bit) — leads; STS/NLI flags disagreements ----
LLM_MODEL_ID = "Qwen/Qwen3-1.7B"   # 8B needs ~16 GB download + >12 GB CPU RAM to load,
                                   # which OOMs free Colab. 1.7B downloads ~3.4 GB & runs fp16.
LLM_ENABLED  = True       # ON, but now BATCHED (see 3.3): ~6-10 min on a T4, not 40.
LLM_BATCH_SIZE = 8        # 1.7B is small -> batch 8 is comfortable on a T4 (lower if OOM).
LLM_GRADE_TOKENS = 64     # max tokens per grade (output is just a score + a few words)
#  Quality vs cost on a FREE T4 (download is one-time per session):
#    Qwen3-1.7B fp16  -> ~3.4 GB dl, ~loads in seconds, fast       (default)
#    Qwen3-4B  fp16   -> ~8 GB dl, tight on 12.7 GB CPU RAM, slower
#    Qwen3-8B  4-bit  -> ~16 GB dl, OOMs free Colab CPU RAM         (needs Colab Pro)
#  To skip re-downloading on every restart, cache to Drive:
#      import os; os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
LLM_WEIGHT   = 0.5        # LLM share when LLM & engine AGREE. 0.5 = equal partners.
                          # (8B can lead at 0.7; a 1.7B should not outvote the engine.)
LLM_DISAGREE = 0.34       # |llm - engine| above this -> 50/50 blend + flag [review]
                          # (the engine no longer gets discarded on disagreement)
LLM_4BIT     = False      # fp16 for a small model: faster than 4-bit AND fits the T4 easily.
                          # (Keep 4-bit ONLY for >=7B models on a >=24 GB GPU.)

# ---- Mapping method: which mechanism assigns each answer segment to its question ----
#   "sbert"    -> STS cross-encoder scores + monotonic (in-order) DP   [default here]
#   "llm"      -> Qwen reads the questions + answers and assigns them
#   "sequence" -> deterministic booklet-order + part-letter + OCR number only (no model)
MAP_METHOD = "marker"   # marker-driven: the student's written Q-number wins; SBERT only fills numberless blocks

# ---- Student-sheet page skipping ----
SKIP_LEADING_PAGES = 2    # MMD booklet: page 1 cover + page 2 instructions (no OCR)
BLANK_INK_FRAC     = 0.004  # handwriting-ink fraction below this -> blank page (skipped)
# blank-detect tuning (strip these printed artifacts BEFORE measuring ink):
BARCODE_BAND_FRAC  = 0.12   # top band holding the barcode
FOOTER_BAND_FRAC   = 0.04   # bottom band holding the [page-no]
MARGIN_LEFT_FRAC   = 0.09   # left double-rule margin column
MARGIN_RIGHT_FRAC  = 0.04   # right dotted margin
LINE_LEN_FRAC      = 0.45   # printed rules run ~this fraction of the page width/height
KEY_SKIP_LEADING   = 0      # set to 2 if your answer KEY is in the same booklet format
KEY_SKIP_BLANKS    = True


### 1.3 Preprocessing (deskew → grayscale → adaptive threshold → denoise)

In [ ]:
# ------------------------------------------------------------------ #
#  Deskew via OpenCV Hough transform (replaces the old MSER/PCA path)  #
# ------------------------------------------------------------------ #
# Why this is better on ruled answer sheets:
#   * Printed rules + handwritten text lines are long, near-horizontal
#     structures — exactly what HoughLinesP detects reliably.
#   * Taking the MEDIAN angle of those lines is robust to outliers.
#   * A dead-zone (DESKEW_MIN_ANGLE) keeps a straight page at 0°, so we
#     NEVER introduce skew where there was none (the old failure mode).
# The estimator returns the angle to APPLY directly via getRotationMatrix2D
# (positive de-rotates the page).

def _projection_skew(gray, angle_range=DESKEW_RANGE, step=DESKEW_STEP):
    """Fallback: maximise row-projection variance over a small angle sweep."""
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h_im, w_im = th.shape
    scale = 1000.0 / max(h_im, w_im)
    if scale < 1.0:
        th = cv2.resize(th, (int(w_im * scale), int(h_im * scale)),
                        interpolation=cv2.INTER_NEAREST)
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-angle_range, angle_range + step, step):
        rot = nd_rotate(th, a, reshape=False, order=0)
        proj = np.sum(rot, axis=1, dtype=np.float64)
        score = float(np.sum(np.diff(proj) ** 2))
        if score > best_score:
            best_score, best_angle = score, float(a)
    return best_angle


def estimate_skew_hough(rgb,
                        angle_range=DESKEW_RANGE,
                        min_angle=DESKEW_MIN_ANGLE,
                        hough_threshold=HOUGH_THRESHOLD,
                        min_len_fraction=HOUGH_MIN_LEN_FR,
                        max_gap=HOUGH_MAX_GAP,
                        min_lines=HOUGH_MIN_LINES,
                        dilate_w=HOUGH_DILATE_W):
    """
    Estimate document skew with the OpenCV probabilistic Hough transform.
    Returns (angle_deg_to_apply, n_lines, method).
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h_im, w_im = gray.shape

    # 1) binarise ink (and rules) -> white on black
    th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # 2) fuse glyphs into horizontal streaks so each text line is one long run
    if dilate_w and dilate_w > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (int(dilate_w), 1))
        th = cv2.dilate(th, kernel, iterations=1)

    # 3) edges -> probabilistic Hough lines
    edges = cv2.Canny(th, 50, 150, apertureSize=3)
    min_len = max(40, int(w_im * min_len_fraction))
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180.0, hough_threshold,
                            minLineLength=min_len, maxLineGap=max_gap)

    # 4) keep near-horizontal lines, take the median angle
    angles = []
    if lines is not None:
        for ln in lines:
            x1, y1, x2, y2 = ln[0]
            ang = np.degrees(np.arctan2(y2 - y1, x2 - x1))
            if ang < -90: ang += 180
            if ang >  90: ang -= 180
            if abs(ang) <= angle_range:
                angles.append(ang)

    if len(angles) >= min_lines:
        angle = float(np.median(angles))
        if abs(angle) < min_angle:
            return 0.0, len(angles), "hough(straight)"
        return float(np.clip(angle, -angle_range, angle_range)), len(angles), "hough"

    # 5) fallback: projection-profile scan
    angle = _projection_skew(gray, angle_range, DESKEW_STEP)
    if abs(angle) < min_angle:
        return 0.0, len(angles), "projection(straight)"
    return float(angle), len(angles), "projection_fallback"


def deskew_rgb(rgb):
    """Deskew an RGB image using the Hough-transform angle estimate."""
    angle, n_lines, method = estimate_skew_hough(rgb)
    if abs(angle) < 1e-3:
        return rgb, 0.0, n_lines, method      # straight -> skip warpAffine entirely
    h, w = rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    rotated = cv2.warpAffine(rgb, M, (w, h),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle, n_lines, method


# ------------------------------------------------------------------ #
#  Adaptive Gaussian Threshold (on gray of deskewed image)            #
# ------------------------------------------------------------------ #
def adaptive_gaussian_threshold(gray):
    bs = ADAPT_BLOCK if ADAPT_BLOCK % 2 == 1 else ADAPT_BLOCK + 1
    return cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        bs, ADAPT_C
    )


# ------------------------------------------------------------------ #
#  Denoise: remove tiny isolated ink specks                           #
# ------------------------------------------------------------------ #
def denoise(binary, min_area=DENOISE_MIN_AREA):
    if min_area <= 0:
        return binary
    ink = (binary == 0).astype(np.uint8)  # ink (black) -> 1
    n, labels, stats, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    keep = np.zeros(n, dtype=bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
    return np.where(keep[labels], 0, 255).astype(np.uint8)


# ------------------------------------------------------------------ #
#  Full pipeline: returns every intermediate stage                    #
# ------------------------------------------------------------------ #
def preprocess(rgb):
    """
    Steps:
      1. Hough-transform deskew on the colour image (with projection fallback)
      2. Grayscale conversion of the deskewed image
      3. Adaptive Gaussian thresholding
      4. Denoise (remove tiny blobs)
    """
    # Step 1 — Hough deskew
    deskewed, angle, n_lines, method = deskew_rgb(rgb)

    # Step 2 — grayscale (only after deskew)
    gray = cv2.cvtColor(deskewed, cv2.COLOR_RGB2GRAY)

    # Step 3 — Adaptive Gaussian threshold
    binary = adaptive_gaussian_threshold(gray)

    # Step 4 — Denoise
    den = denoise(binary)

    return {
        "original":      rgb,
        "deskewed":      deskewed,
        "gray":          gray,
        "threshold":     binary,
        "denoised":      den,
        "angle":         angle,
        "n_lines":       n_lines,
        "deskew_method": method,
    }

### 1.4 Load Chandra OCR 2 (fp16 on T4 + SDPA + self-test)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from chandra.prompts import PROMPT_MAPPING
from chandra.output import parse_markdown
from chandra.model.util import scale_to_fit, detect_repeat_token

if not torch.cuda.is_available():
    print("WARNING: no CUDA GPU detected. Chandra OCR 2 (5B) is very slow on CPU.")
    print("On Colab: Runtime > Change runtime type > T4 GPU")

# If the model is gated, accept the licence on HuggingFace and log in:
# from huggingface_hub import login; login()

def _gpu_pre_ampere():
    """True on T4/Turing-class GPUs (compute capability < 8.0): no native bf16."""
    return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8

def _resolve_ocr_dtype():
    if OCR_DTYPE == "fp16": return torch.float16
    if OCR_DTYPE == "bf16": return torch.bfloat16
    # auto: bf16 has NO tensor-core support on T4 (sm75) -> emulated & slow.
    # fp16 uses the T4's native fp16 tensor cores. On A100/H100+ keep bf16.
    return torch.float16 if _gpu_pre_ampere() else torch.bfloat16

def _load_chandra(dtype):
    # Pin to GPU 0 (fail loudly instead of silently offloading layers to CPU,
    # which device_map="auto" can do under memory pressure -> 10x slower OCR).
    dm = {"": 0} if torch.cuda.is_available() else "auto"
    last_err = None
    for kw in (dict(dtype=dtype, attn_implementation="sdpa"),  # sdpa: fastest attn on T4 (FA2 needs Ampere+)
               dict(dtype=dtype),
               dict(torch_dtype=dtype)):                       # transformers <5 fallback
        try:
            m = AutoModelForImageTextToText.from_pretrained(MODEL_ID, device_map=dm, **kw)
            break
        except (TypeError, ValueError) as e:
            last_err = e
    else:
        raise last_err
    m.eval()
    m.processor = AutoProcessor.from_pretrained(MODEL_ID)
    tok = m.processor.tokenizer
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return m

def _fp16_self_test(m):
    """~2 s sanity check: fp16 can overflow on some checkpoints. Render a tiny
    synthetic strip, OCR it, and make sure the output isn't empty/degenerate."""
    from PIL import ImageDraw
    img = Image.new("RGB", (384, 96), "white")
    ImageDraw.Draw(img).text((10, 35), "OCR SELF TEST 123", fill="black")
    conv = [[{"role": "user", "content": [
        {"type": "image", "image": scale_to_fit(img)},
        {"type": "text",  "text": PROMPT_MAPPING[PROMPT_TYPE]},
    ]}]]
    x = m.processor.apply_chat_template(
        conv, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True).to(m.device)
    try:
        with torch.inference_mode():
            out = m.generate(**x, max_new_tokens=24, do_sample=False,
                             pad_token_id=m.processor.tokenizer.pad_token_id)
        t = m.processor.tokenizer.decode(
            out[0, x.input_ids.shape[1]:], skip_special_tokens=True).strip()
    except RuntimeError as e:
        return False, f"error: {str(e)[:80]}"
    if not t or len(set(t)) <= 2:
        return False, f"degenerate output: {t[:40]!r}"
    return True, t[:40]

_dtype = _resolve_ocr_dtype()
print(f"Loading Chandra OCR 2 in {str(_dtype).split('.')[-1]}"
      + ("  (T4-class GPU: native fp16 tensor cores; bf16 here is emulated and slow)"
         if _dtype == torch.float16 else ""))
model = _load_chandra(_dtype)
print("attention impl:", getattr(model.config, "_attn_implementation", "default"))

if _dtype == torch.float16:
    ok, info = _fp16_self_test(model)
    if ok:
        print(f"fp16 self-test OK -> {info!r}")
    else:
        print(f"fp16 self-test FAILED ({info}) -> reloading in bf16 (slower but safe)")
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        model = _load_chandra(torch.bfloat16)

print("Chandra OCR 2 loaded.")


### 1.5 OCR runner + GPU shuffle (batched, loop-guarded) + **Chandra unload**

In [ ]:
import time
from transformers import StoppingCriteria, StoppingCriteriaList

def load_pages(path, dpi=DPI):
    """Render every PDF page (or a single image file) to an RGB PIL Image."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        doc = fitz.open(path)
        mat = fitz.Matrix(dpi / 72.0, dpi / 72.0)
        imgs = []
        for page in doc:
            pix = page.get_pixmap(matrix=mat, alpha=False)
            imgs.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
        doc.close()
        return imgs
    return [Image.open(path).convert("RGB")]

def _cap_image(pil_img, max_side=MAX_OCR_SIDE):
    w, h = pil_img.size
    longest = max(w, h)
    if longest <= max_side:
        return pil_img
    scale = max_side / longest
    return pil_img.resize((max(1, int(w*scale)), max(1, int(h*scale))), Image.LANCZOS)

def _clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

# ---- GPU shuffle: grading models <-> CPU so Chandra always has the VRAM ----
def free_gpu_for_ocr():
    """Move every loaded grading model to CPU and clear the CUDA cache."""
    moved = []
    g = globals()
    enc = g.get("encoder");  sts = g.get("_STS");  nli = g.get("_NLI")
    try:
        if enc is not None: enc.to("cpu"); moved.append("BGE")
    except Exception: pass
    try:
        if sts is not None: sts.model.to("cpu"); moved.append("STS")
    except Exception: pass
    try:
        if nli is not None: nli.model.to("cpu"); nli.device = torch.device("cpu"); moved.append("NLI")
    except Exception: pass
    _clear_gpu()
    if moved: print("Moved to CPU for OCR:", ", ".join(moved))

def grading_models_to_gpu():
    """Move loaded grading models back to GPU (no-op without CUDA)."""
    if not torch.cuda.is_available(): return
    g = globals(); moved = []
    enc = g.get("encoder");  sts = g.get("_STS");  nli = g.get("_NLI")
    try:
        if enc is not None: enc.to("cuda"); moved.append("BGE")
    except Exception: pass
    try:
        if sts is not None: sts.model.to("cuda"); moved.append("STS")
    except Exception: pass
    try:
        if nli is not None: nli.model.to("cuda"); nli.device = torch.device("cuda:0"); moved.append("NLI")
    except Exception: pass
    if moved: print("Grading models on GPU:", ", ".join(moved))

# ------------------------------------------------------------------ #
#  Repetition loop guard                                              #
#  Chandra's HF path has NO protection against degenerate repeat     #
#  loops and a 12,384-token default cap -> one bad page = minutes.   #
#  This StoppingCriteria aborts a sequence as soon as its tail is a  #
#  repeating cycle (same scaling rule as chandra's own               #
#  detect_repeat_token, but on token ids, while generating).         #
# ------------------------------------------------------------------ #
class _LoopStop(StoppingCriteria):
    def __init__(self, prompt_len, batch_size, window=256, max_cycle=96, every=16, min_gen=96):
        self.p = prompt_len
        self.window, self.max_cycle, self.every, self.min_gen = window, max_cycle, every, min_gen
        self.calls = 0
        self.looping = [False] * batch_size

    @staticmethod
    def _tail_loops(ids, max_cycle):
        L = len(ids)
        for c in range(1, min(max_cycle, L // 3) + 1):
            need = int(4 * (1 + 3.0 / c)) + 1      # short cycles need more repeats
            if c * need > L: continue
            unit = ids[-c:]
            reps, pos = 1, L - 2 * c
            while pos >= 0 and ids[pos:pos + c] == unit:
                reps += 1
                if reps >= need: return True
                pos -= c
        return False

    def __call__(self, input_ids, scores, **kw):
        self.calls += 1
        flags = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        for r, lp in enumerate(self.looping): flags[r] = lp
        if self.calls % self.every: return flags
        gen_len = input_ids.shape[1] - self.p
        if gen_len < self.min_gen: return flags
        tail = input_ids[:, -min(self.window, gen_len):].tolist()
        for r, row in enumerate(tail):
            if not self.looping[r] and self._tail_loops(row, self.max_cycle):
                self.looping[r] = True
                flags[r] = True
        return flags

# ------------------------------------------------------------------ #
#  Batched Chandra engine (replaces chandra.model.hf.generate_hf)    #
#  Same prompts / chat template / parsing — adds: page batching,     #
#  sane token cap, loop guard, per-batch tok/s timing.               #
# ------------------------------------------------------------------ #
def _eos_ids():
    eos = model.generation_config.eos_token_id
    eos = [eos] if isinstance(eos, int) else list(eos or [])
    im_end = model.processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end is not None and im_end not in eos: eos.append(im_end)
    return eos

def _build_inputs(pil_images, prompt_type=PROMPT_TYPE):
    convs = [[{"role": "user", "content": [
                {"type": "image", "image": scale_to_fit(im)},   # chandra's own 28-px grid snap
                {"type": "text",  "text": PROMPT_MAPPING[prompt_type]},
             ]}] for im in pil_images]
    inputs = model.processor.apply_chat_template(
        convs, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True)
    return inputs.to(model.device)

def _generate_batch(pil_images, max_new_tokens=None, sample=False, quiet=False):
    if max_new_tokens is None: max_new_tokens = MAX_NEW_TOKENS
    t0 = time.time()
    inputs = _build_inputs(pil_images)
    p_len = inputs.input_ids.shape[1]
    crit = StoppingCriteriaList([_LoopStop(p_len, len(pil_images))]) if LOOP_GUARD else None
    gen_kw = dict(max_new_tokens=max_new_tokens,
                  eos_token_id=_eos_ids(),
                  pad_token_id=model.processor.tokenizer.pad_token_id,
                  stopping_criteria=crit)
    if sample: gen_kw.update(do_sample=True, temperature=0.3, top_p=0.95)
    else:      gen_kw.update(do_sample=False)
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    texts, ntok = [], 0
    for row in out:
        gen = row[p_len:]                       # left padding -> same offset for every row
        ntok += int(gen.shape[0])
        texts.append(model.processor.tokenizer.decode(gen, skip_special_tokens=True))
    dt = time.time() - t0
    if not quiet:
        print(f"    [chandra] {len(pil_images)} page(s) | {ntok} tok | {dt:.1f}s | {ntok/max(dt,1e-9):.1f} tok/s")
    return texts

def _finalize(raw):
    try:    return parse_markdown(raw)
    except Exception:
        return raw or ""

def _trim_loop(text):
    """If the tail of the OCR text is a repeating cycle, cut the repeats.
    Exponential + binary search for the SMALLEST cut that silences chandra's
    detector, so real answer content before the loop is preserved (at most a
    couple of residual loop units remain — normalize_ocr_lines drops those)."""
    if not text or not detect_repeat_token(text):
        return text, False
    lo, hi = 0, 50
    while hi < len(text) and detect_repeat_token(text, cut_from_end=hi):
        lo, hi = hi, min(len(text), hi * 2)
    while lo + 25 < hi:
        mid = (lo + hi) // 2
        if detect_repeat_token(text, cut_from_end=mid):
            lo = mid
        else:
            hi = mid
    return text[:max(0, len(text) - hi)].rstrip(), True

def run_chandra_page(np_img, prompt_type=PROMPT_TYPE, max_side=MAX_OCR_SIDE):
    """Single-page wrapper kept for backwards compatibility (OOM-safe)."""
    pil_raw = Image.fromarray(np_img).convert("RGB")
    pil_img = _cap_image(pil_raw, max_side)
    try:
        raw = _generate_batch([pil_img])[0]
    except torch.cuda.OutOfMemoryError:
        print(f"    [OOM at {max_side}px] retrying at {max_side//2}px ...")
        _clear_gpu()
        raw = _generate_batch([_cap_image(pil_raw, max_side // 2)])[0]
    txt, looped = _trim_loop(_finalize(raw))
    return txt

def ocr_document(path, show_preprocess=False, pages_per_batch=None):
    """PDF/image -> list of per-page OCR markdown strings (with preprocessing).

    v6: pages are OCR'd PAGES_PER_BATCH at a time (decode is memory-bandwidth
    bound, so 2 pages per forward pass ~= 1.7-1.9x throughput on a T4), the
    CUDA cache is no longer nuked between every page, and repetition loops are
    aborted early + trimmed (+ optionally retried once with light sampling)."""
    if pages_per_batch is None: pages_per_batch = PAGES_PER_BATCH
    free_gpu_for_ocr()                       # park grading models first
    pages = load_pages(path, dpi=DPI)
    print(f"Loaded {len(pages)} page(s) at {DPI} DPI from {os.path.basename(path)}")

    prepped = []
    for i, img in enumerate(pages):
        st = preprocess(np.array(img))
        print(f"  Page {i+1}/{len(pages)}: skew={st['angle']:+.2f}° ({st['deskew_method']})")
        if show_preprocess:
            fig, axes = plt.subplots(1, 2, figsize=(10, 6))
            axes[0].imshow(st["original"]); axes[0].set_title("Original"); axes[0].axis("off")
            axes[1].imshow(st["denoised"], cmap="gray"); axes[1].set_title("Preprocessed"); axes[1].axis("off")
            plt.tight_layout(); plt.show()
        prepped.append(_cap_image(Image.fromarray(st["denoised"]).convert("RGB"), MAX_OCR_SIDE))

    texts = [None] * len(prepped)
    bs = max(1, int(pages_per_batch))
    t_doc = time.time()
    i = 0
    while i < len(prepped):
        chunk = prepped[i:i + bs]
        try:
            raws = _generate_batch(chunk)
        except torch.cuda.OutOfMemoryError:
            _clear_gpu()
            if len(chunk) > 1:
                print("    [OOM] dropping to 1 page per batch")
                bs = 1
                continue
            print(f"    [OOM] page {i+1}: retrying at half size")
            half = chunk[0]
            half = half.resize((max(1, half.width // 2), max(1, half.height // 2)), Image.LANCZOS)
            raws = _generate_batch([half])
        for j, raw in enumerate(raws):
            txt, looped = _trim_loop(_finalize(raw))
            if looped:
                print(f"    page {i+j+1}: repetition loop detected -> trimmed tail", end="")
                if RETRY_ON_LOOP:
                    print(" + one retry with light sampling")
                    r2, _ = _trim_loop(_finalize(_generate_batch([prepped[i+j]], sample=True, quiet=True)[0]))
                    if len(r2) > len(txt): txt = r2
                else:
                    print()
            texts[i + j] = txt
            print(f"    page {i+j+1}: -> {len(txt)} chars, {len(txt.split())} words")
        i += len(chunk)
    _clear_gpu()
    dt = time.time() - t_doc
    print(f"OCR done: {len(prepped)} page(s) in {dt:.1f}s  ({dt/max(1,len(prepped)):.1f}s/page)")
    return texts

def read_any(path):
    """ANY uploaded doc -> list of per-page text strings (OCR only when needed)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".pdf", ".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp"):
        return ocr_document(path)
    if ext == ".docx":
        return ["\n".join(load_docx_lines(path))]      # defined in 1.6
    return [open(path, "r", encoding="utf-8", errors="replace").read()]

print("OCR runner ready (v6: batched + loop-guarded).")

# ---- v8: free Chandra entirely after OCR so the 8B grader + STS/NLI fit one L4 ----
def unload_chandra():
    """Drop the Chandra model from VRAM once OCR is done. It reloads lazily on the
    next OCR call. This is what lets a ~10 GB OCR model and an 8B grader share 24 GB."""
    global model
    try:
        m = model; model = None; del m
    except Exception:
        model = None
    _clear_gpu()
    print("Chandra unloaded from GPU (reloads on next OCR) — room for graders + Qwen.")

def unload_llm():
    """Drop the Qwen grader from VRAM (so Chandra can reload on a small GPU)."""
    try:
        if _LLM.get("model") is not None:
            _LLM["model"] = None; _LLM["ready"] = False
    except Exception:
        pass
    _clear_gpu()

def ensure_chandra():
    """Reload Chandra if it was unloaded. Frees the grading models AND the Qwen grader
    first, so OCR never has to share the GPU with them (prevents OOM on 16 GB cards)."""
    global model
    if globals().get("model") is None:
        try: free_gpu_for_ocr()      # park BGE/STS/NLI on CPU
        except Exception: pass
        try: unload_llm()            # drop Qwen from GPU
        except Exception: pass
        _clear_gpu()
        print("Reloading Chandra OCR 2 ...")
        model = _load_chandra(_resolve_ocr_dtype())
        print("Chandra reloaded.")

# ---- v8: return RAW Chandra output (pre-parse) so we can build BOTH grading lines
#          AND the structured HTML from ONE OCR pass ----
def ocr_images_raw(prepped_imgs, pages_per_batch=None):
    """Preprocessed PIL images -> list of RAW Chandra strings (one OCR pass)."""
    if pages_per_batch is None: pages_per_batch = PAGES_PER_BATCH
    raws = [None]*len(prepped_imgs); bs = max(1, int(pages_per_batch)); i = 0
    while i < len(prepped_imgs):
        chunk = prepped_imgs[i:i+bs]
        try:
            rs = _generate_batch(chunk)
        except torch.cuda.OutOfMemoryError:
            _clear_gpu()
            if len(chunk) > 1:
                bs = 1; continue
            half = chunk[0].resize((max(1,chunk[0].width//2), max(1,chunk[0].height//2)), Image.LANCZOS)
            rs = _generate_batch([half])
        for j, raw in enumerate(rs):
            txt, _ = _trim_loop(raw)
            raws[i+j] = txt
        i += len(chunk)
    return raws


### 1.5b Page skipping — leading instruction pages + **blank pages by handwriting ink**
Measures handwriting ink only: the barcode band, footer page-number, margin rules and printed ruled/dotted lines are stripped first (they're the ~3.5–4.4 % ink floor), then what's left is the handwriting.

In [ ]:
import numpy as np, cv2

def _suppress_printed_lines(bw, line_len_frac, axis):
    h, w = bw.shape
    if axis == "h":
        bridge = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 1))
        ln_k   = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, int(w*line_len_frac)), 1))
        grow   = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
    else:
        bridge = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 9))
        ln_k   = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, int(h*line_len_frac))))
        grow   = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1))
    bridged = cv2.dilate(bw, bridge)
    lines   = cv2.morphologyEx(bridged, cv2.MORPH_OPEN, ln_k)
    lines   = cv2.dilate(lines, grow)
    lines   = cv2.bitwise_and(lines, bw)
    return cv2.subtract(bw, lines)

def handwriting_ink_fraction(page_rgb):
    """Fraction of the page that is handwriting ink, after removing barcode band,
    margins, footer and printed ruled/dotted lines."""
    arr = np.asarray(page_rgb)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY) if arr.ndim == 3 else arr
    h, w = gray.shape
    bw = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 35, 15)
    bw[:int(h*BARCODE_BAND_FRAC), :] = 0
    bw[int(h*(1-FOOTER_BAND_FRAC)):, :] = 0
    bw[:, :int(w*MARGIN_LEFT_FRAC)] = 0
    bw[:, int(w*(1-MARGIN_RIGHT_FRAC)):] = 0
    bw = _suppress_printed_lines(bw, LINE_LEN_FRAC, "h")
    bw = _suppress_printed_lines(bw, LINE_LEN_FRAC, "v")
    bw = cv2.medianBlur(bw, 3)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT,(2,2)))
    return float(np.count_nonzero(bw)) / float(bw.size)

def is_blank_page(page_rgb):
    return handwriting_ink_fraction(page_rgb) < BLANK_INK_FRAC

def select_pages(path, skip_leading=0, skip_blanks=True, dpi=None):
    """Render a PDF/image, drop the first `skip_leading` pages, drop blanks.
    Returns (kept_pil_images, kept_original_1based_indices)."""
    pages = load_pages(path, dpi=dpi or DPI)
    kept, idx = [], []
    for i, im in enumerate(pages):
        if i < skip_leading:
            print(f"  page {i+1}: skipped (leading instruction page)"); continue
        if skip_blanks and is_blank_page(im):
            print(f"  page {i+1}: skipped (blank — hw-ink {handwriting_ink_fraction(im):.4f})"); continue
        kept.append(im); idx.append(i+1)
    print(f"  -> kept {len(kept)}/{len(pages)} page(s): {idx}")
    return kept, idx

def calibrate_blank_threshold(path, skip_leading=0):
    """Print the handwriting-ink fraction per page so you can set BLANK_INK_FRAC.
    Run once on a known sheet: written pages should sit clearly above blank ones."""
    pages = load_pages(path, dpi=DPI)
    print(f"{'page':>4} | {'hw_ink':>8} | guess")
    for i, im in enumerate(pages):
        f = handwriting_ink_fraction(im)
        tag = "(leading)" if i < skip_leading else ("BLANK" if f < BLANK_INK_FRAC else "written")
        print(f"{i+1:>4} | {f:>8.4f} | {tag}")
print("Page-skip + blank-by-handwriting-ink ready.")


### 1.5c Structured-HTML export — equations (MathJax, offline) + diagrams (cropped & embedded)
Lifted from the equation/diagram pipeline. Chandra-2's `parse_layout` gives typed blocks; `Equation-Block`/`Chemical-Block` → MathJax, `Figure`/`Diagram` → bbox-cropped base64 PNG. One self-contained `.html`.

In [ ]:
from chandra.output import parse_layout, parse_html
from collections import Counter
import io, base64, html as _htmllib, urllib.request

VISUAL_LABELS   = {"Figure", "Image", "Diagram"}   # blocks we crop & embed as pictures

_BLOCK_GROUPS = {
    "Text":           ["Text","Section-Header","Caption","Footnote","List-Group",
                       "Bibliography","Table-Of-Contents","Page-Header","Page-Footer",
                       "Form","Code-Block","Complex-Block"],
    "Math equation":  ["Equation-Block"],
    "Chemical":       ["Chemical-Block"],
    "Table":          ["Table"],
    "Figure / image": ["Figure","Image","Diagram"],
}
_LABEL2GROUP = {lab: g for g, labs in _BLOCK_GROUPS.items() for lab in labs}


# ---------- math / chemistry: Chandra tags -> MathJax delimiters ----------
def _to_mathjax(body):
    def _m(m):
        attrs, tex = m.group(1), _htmllib.unescape(m.group(2))
        block = ("display" in attrs and "block" in attrs)
        return ("\\[" + tex + "\\]") if block else ("\\(" + tex + "\\)")
    body = re.sub(r"<math\b([^>]*)>(.*?)</math>", _m, body, flags=re.S|re.I)
    def _c(m):
        s = _htmllib.unescape(m.group(1)).strip()
        return "\\(" + (s if s.startswith("\\ce") else "\\ce{"+s+"}") + "\\)"
    body = re.sub(r"<chem\b[^>]*>(.*?)</chem>", _c, body, flags=re.S|re.I)
    return body


# ---------- crop a layout block out of the page & return a base64 PNG ----------
def _crop_b64(page_img, bbox, pad=6):
    w, h = page_img.size
    x0, y0, x1, y1 = bbox
    x0, y0 = max(0, x0 - pad), max(0, y0 - pad)
    x1, y1 = min(w, x1 + pad), min(h, y1 + pad)
    if (x1 - x0) < 6 or (y1 - y0) < 6:
        return None, None
    crop = page_img.crop((x0, y0, x1, y1))
    buf = io.BytesIO(); crop.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode(), crop


def _is_barcode_block(bbox, page_img, desc):
    """The MMD booklet prints a barcode band at the top of every page; Chandra
    crops it as a wide, short, captionless figure. Detect & drop those."""
    x0, y0, x1, y1 = bbox; W, H = page_img.size
    w, h = (x1 - x0), (y1 - y0)
    if w <= 0 or h <= 0: return False
    wide     = (w / h) >= 4.0
    short    = h < 0.06 * H
    near_top = y0 < 0.12 * H
    no_text  = len(re.sub(r"[^A-Za-z]", "", desc)) < 8
    return wide and short and near_top and no_text

def _block_to_html(block, page_img, figures, page_no):
    """Turn one layout block into HTML. Visual blocks get cropped & embedded."""
    label   = block.label or "block"
    content = block.content or ""
    if label in VISUAL_LABELS:
        desc = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", content)).strip()
        if _is_barcode_block(block.bbox, page_img, desc):       # drop barcode strips
            return ""
        b64, crop = _crop_b64(page_img, block.bbox)
        if b64 is not None:
            figures.append((page_no, crop, desc))            # for inline Colab preview
            cap = f'<figcaption>{_htmllib.escape(desc)}</figcaption>' if desc else ""
            return (f'<figure class="ocr-figure"><img src="{b64}" '
                    f'alt="{_htmllib.escape(desc)}"/>{cap}</figure>')
        return f'<div class="figure-note">[ {label}: {_htmllib.escape(desc)} ]</div>' if desc else ""
    # text / table / math / list: keep the block content as-is (wrap bare text)
    c = content.strip()
    if c and not re.search(r"<\w+", c):
        c = f"<p>{c}</p>"
    return c


def _page_body(raw, page_img, figures, page_no):
    """Build one page's HTML body from layout blocks (so we can embed figures)."""
    try:
        blocks = parse_layout(raw, page_img)
    except Exception:
        blocks = []
    if not blocks:                                           # fallback if no layout divs
        return parse_html(raw, include_images=False) or _htmllib.escape(raw)
    return "\n".join(_block_to_html(b, page_img, figures, page_no) for b in blocks)


def summarize_blocks(raw_html, used_img):
    try:
        blocks = parse_layout(raw_html, used_img)
    except Exception:
        blocks = []
    grp = Counter()
    for b in blocks:
        grp[_LABEL2GROUP.get(b.label, "Other")] += 1
    n_math  = len(re.findall(r"<math\b",  raw_html, flags=re.I))
    n_chem  = len(re.findall(r"<chem\b",  raw_html, flags=re.I))
    n_table = len(re.findall(r"<table\b", raw_html, flags=re.I))
    return grp, n_math, n_chem, n_table


# ---------- HTML doc (MathJax SVG inlined -> renders with no internet) ----------
_MJ_CORE = None
def _mathjax_core():
    global _MJ_CORE
    if _MJ_CORE is None:
        url = "https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-svg.js"
        print("    fetching MathJax once to inline into the HTML ...")
        _MJ_CORE = urllib.request.urlopen(url, timeout=180).read().decode("utf-8")
        _MJ_CORE = _MJ_CORE.replace("</script>", "<\\/script>")
    return _MJ_CORE

def build_html_document(body_html, title="Chandra OCR output"):
    body = _to_mathjax(body_html)
    needs_chem = ("\\ce" in body)
    cfg_pkg  = "packages: {'[+]': ['mhchem']}," if needs_chem else ""
    cfg_load = "loader: { load: ['[tex]/mhchem'] }," if needs_chem else ""
    core = _mathjax_core()
    head = (
        '<!DOCTYPE html>\n<html lang="en"><head>\n'
        '<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n'
        '<title>' + _htmllib.escape(title) + '</title>\n'
        '<style>\n'
        '  body{font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;max-width:880px;margin:32px auto;padding:0 22px;line-height:1.6;color:#161616;}\n'
        '  table{border-collapse:collapse;margin:14px 0;width:100%;}\n'
        '  th,td{border:1px solid #c9c9c9;padding:6px 10px;text-align:left;vertical-align:top;} th{background:#f2f2f2;}\n'
        '  .ocr-page{padding:20px 0;border-bottom:1px dashed #ddd;}\n'
        '  .ocr-page-num{color:#999;font-size:12px;letter-spacing:.06em;text-transform:uppercase;margin-bottom:8px;}\n'
        '  figure.ocr-figure{margin:14px 0;text-align:center;}\n'
        '  figure.ocr-figure img{max-width:100%;border:1px solid #e2e2e2;border-radius:6px;}\n'
        '  figcaption{color:#777;font-size:13px;font-style:italic;margin-top:6px;}\n'
        '  .figure-note{color:#888;font-style:italic;background:#fafafa;border:1px dashed #ddd;padding:6px 10px;border-radius:4px;margin:10px 0;}\n'
        '  mjx-container{overflow-x:auto;overflow-y:hidden;}\n'
        '</style>\n'
        '<script>\nwindow.MathJax = {\n'
        "  tex: { inlineMath:[['\\\\(','\\\\)'],['$','$']], displayMath:[['\\\\[','\\\\]'],['$$','$$']], " + cfg_pkg + " },\n"
        '  ' + cfg_load + '\n'
        "  svg: { fontCache:'local' }\n"
        '};\n</script>\n'
        '<script>' + core + '</script>\n'
        '</head><body>\n'
    )
    return head + body + '\n</body></html>'

# ---- v8: build the structured HTML from RAW strings we already OCR'd (no 2nd pass) ----
def structured_html_from_raws(raws, imgs, page_indices, out_html, title):
    page_bodies, figures, totals = [], [], Counter()
    for raw, img, pno in zip(raws, imgs, page_indices):
        body = _page_body(raw, img, figures, pno)
        page_bodies.append(f'<div class="ocr-page"><div class="ocr-page-num">Page {pno}</div>\n{body}\n</div>')
        grp, *_ = summarize_blocks(raw, img)
        totals += grp
    doc = build_html_document("\n".join(page_bodies), title=title)
    # safety net: convert any <math>/<chem> tags _to_mathjax missed into MathJax delimiters
    doc = re.sub(r"<chem[^>]*>(.*?)</chem>", lambda m: r"\(\ce{" + m.group(1).strip() + r"}\)", doc, flags=re.S)
    doc = re.sub(r"<math[^>]*>(.*?)</math>", lambda m: r"\(" + m.group(1).strip() + r"\)", doc, flags=re.S)
    doc = re.sub(r"</?math[^>]*>|</?chem[^>]*>", "", doc)   # drop any stragglers
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(doc)
    blk = ", ".join(f"{k}={v}" for k, v in totals.items()) or "(none)"
    print(f"[OK] wrote {out_html}  ({len(doc)/1e6:.2f} MB, opens offline) | blocks: {blk} | figures: {len(figures)}")
    return out_html, figures
print("Structured-HTML export ready.")


### 1.6 Text bridge — OCR markdown → clean evaluator lines (+ DOCX reader)

In [ ]:
# ============================================================================
#  BRIDGE  —  OCR output  ->  evaluator input  (NO .docx round-trip)
# ----------------------------------------------------------------------------
#  OLD pipeline:  OCR markdown -> pandoc -> .docx -> load_student_lines()
#     The Word round-trip injected noise (wrapper headings, "Page N" / "Source"
#     lines), silently dropped LaTeX math (equations are not returned by
#     python-docx's .text), and re-flowed tables. That degraded both the
#     marker detection (Tier 1) and the scoring -> high accuracy error.
#
#  NEW pipeline (this cell):  OCR markdown -> markdown_to_lines()
#                             -> normalize_ocr_lines() -> student_lines
#     We produce the SAME flat list of text "lines" the evaluator expects,
#     entirely in memory, with no file written and no Word conversion.
# ============================================================================
import re, unicodedata, os

# ---- OCR-artifact normalisation (identical rules to the original evaluator) ----
_CIRCLED = "".join(chr(c) for c in list(range(0x2460, 0x2474)) + list(range(0x2776, 0x277F)))
_ARROWS  = "\u2192\u2193\u21d2\u27a4\u2794\u27f6\u21e8\u21b3\u279c\u2b9c"

_LATEX_ARROW = re.compile(r"\\(?:right|long|Right|Long)?(?:arrow|rightarrow)\b|\\to\b|\\implies\b|\\mapsto\b|\\leadsto\b", re.I)
_LATEX_DROP  = re.compile(r"\\[\(\)\[\]]|\\(?:left|right|text|mathrm|mathbf|mathit|displaystyle|quad|;|,)\b|[{}]")
def _clean_line(s):
    s = "".join(ch for ch in s if ch not in _CIRCLED)        # drop circled-number marks (before NFKC)
    for a in _ARROWS: s = s.replace(a, " ")                  # unicode arrows -> space
    s = _LATEX_ARROW.sub(" -> ", s)                          # \\rightarrow etc -> ->
    s = _LATEX_DROP.sub(" ", s)                              # \\( \\) \\[ \\] \\left \\text{ ... -> space
    s = s.replace("$", " ")                                  # stray math delimiters
    s = unicodedata.normalize("NFKC", s)                     # full-width/ligatures -> ascii
    s = (s.replace("\u2019", "'").replace("\u2018", "'")
           .replace("\u201c", '"').replace("\u201d", '"'))   # curly quotes -> straight
    s = re.sub(r"(?<!\w)X(?!\w)", " ", s)                    # teacher "X" wrong-marks
    return re.sub(r"\s+", " ", s).strip()


# Letterhead / exam-boilerplate lines that are pure noise to the parser.
_BOILER = re.compile(r"(start\s+writing\s+from\s+here|centre\s+for\s+development\s+of\s+advanced|^\s*c-?dac\s*$|answer\s*book|\bmid[\s-]*term\b|\bmid\s*sem\b)", re.I)
def _is_boiler(s):
    if _BOILER.search(s): return True
    dev = sum(1 for ch in s if "\u0900" <= ch <= "\u097f")   # Devanagari (CDAC logo) dominant line
    return bool(dev) and dev >= max(1, len(s.replace(" ", "")) // 2)

def normalize_ocr_lines(lines):
    out = []
    for ln in lines:
        s = _clean_line(ln)
        if not s:
            continue
        if _is_boiler(s):                                   # drop CDAC / 'Start Writing' / 'Mid term'
            continue
        if re.fullmatch(r"\d{1,3}", s):                      # lone page/teacher-mark digit -> drop
            continue
        if re.fullmatch(r"\[\s*\d{1,3}\s*\]", s):              # bracketed page footer [6] -> drop
            continue
        if re.fullmatch(r"[.)\-_=\u2022\u00b7]{1,4}", s):    # lone punctuation/bullet -> drop
            continue
        out.append(s)
    return out

# ---- Markdown -> flat lines (mirrors what the .docx loader used to yield) ----
# The evaluator's old DOCX loader produced: heading/list text WITHOUT the markup
# glyphs, and table rows flattened as "cell | cell". We reproduce that here so
# Tier-1 marker detection ("Q11 ...", "Ans - 2 : ...") and the table handling keep
# working exactly as before — just without the lossy Word conversion.
_TABLE_SEP_RE = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(?:\|\s*:?-{2,}:?\s*)*\|?\s*$")  # |---|---|
_HR_RE        = re.compile(r"^\s*([-*_])\1{2,}\s*$")                                # --- *** ___
_ATX_RE       = re.compile(r"^\s*#{1,6}\s*")                                        # # heading
_BULLET_RE    = re.compile(r"^\s*(?:[-*+\u2022\u00b7])\s+")                           # dash/star bullets ONLY (keep "1."/"2)" markers)
_BQ_RE        = re.compile(r"^\s*>+\s?")                                            # > blockquote
_EMPH_RE      = re.compile(r"(\*\*|__|\*|_|`)")                                     # emphasis / code
_HTML_INLINE  = re.compile(r"</?(?:br|b|i|u|strong|em|p|div|span|sub|sup|small)\s*/?>", re.I)
_MD_LINK      = re.compile(r"\[([^\]]+)\]\([^)]*\)")
_MD_IMG       = re.compile(r"!\[[^\]]*\]\([^)]*\)")

def _flatten_pipe_row(line):
    """'| a | b |' -> 'a | b' (drop empty cells) — matches the old table flattening."""
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    cells = [c for c in cells if c]
    return " | ".join(cells)

def _flatten_html_tables(text):
    """Chandra can emit <table> HTML. Flatten each row to 'a | b' so it reads like
    a table row to the evaluator instead of becoming raw <tag> noise."""
    def repl(m):
        rows = []
        for tr in re.findall(r"<tr[\s\S]*?</tr>", m.group(0), flags=re.I):
            cells = re.findall(r"<t[dh][^>]*>([\s\S]*?)</t[dh]>", tr, flags=re.I)
            cells = [re.sub(r"<[^>]+>", " ", c) for c in cells]          # strip nested tags
            cells = [re.sub(r"\s+", " ", c).strip() for c in cells]
            cells = [c for c in cells if c]
            if cells:
                rows.append(" | ".join(cells))
        return "\n" + "\n".join(rows) + "\n"
    return re.sub(r"<table[\s\S]*?</table>", repl, text, flags=re.I)

def _strip_md_inline(s):
    s = _MD_IMG.sub(" ", s)                 # ![alt](url) -> drop
    s = _MD_LINK.sub(r"\1", s)              # [text](url) -> text
    s = _EMPH_RE.sub("", s)                 # **bold** / *italic* / `code` -> text
    s = _HTML_INLINE.sub(" ", s)            # <br>, <b>, ... -> space
    return s

def markdown_to_lines(md_text):
    """Convert one page of Chandra OCR markdown into flat text lines:
       * <table> HTML and markdown pipe tables -> 'a | b' rows
       * markdown table separators / horizontal rules -> dropped
       * headings (#), bullets (-, *, 1.), blockquotes (>) -> markup stripped, text kept
       * $$math$$ / $math$ delimiters stripped (inner text kept, harmless for this rubric)
       * emphasis / inline-HTML markup stripped
    """
    md_text = _flatten_html_tables(str(md_text))
    md_text = md_text.replace("```", " ").replace("~~~", " ")   # code fences
    md_text = md_text.replace("$$", " ").replace("$", " ")      # math delimiters
    out = []
    for ln in md_text.split("\n"):
        if _TABLE_SEP_RE.match(ln) or _HR_RE.match(ln):
            continue
        s = _BQ_RE.sub("", ln)
        s = _ATX_RE.sub("", s)
        s = _BULLET_RE.sub("", s)
        if s.strip().startswith("|"):       # markdown table row
            s = _flatten_pipe_row(s)
        s = _strip_md_inline(s)
        out.append(s)
    return out

def lines_from_ocr_pages(pages):
    """OCR markdown (list of per-page strings) -> normalized student lines.
    This REPLACES load_student_lines(STUDENT_DOCX) for the student answers.
    Pages are concatenated directly (no '=== PAGE n ===' / '## Page n' markers,
    which were pure noise to the parser)."""
    raw = []
    for txt in pages:
        raw.extend(markdown_to_lines(txt))
    return normalize_ocr_lines(raw)


# DOCX loader — used ONLY if you supply an answer_key.docx (ANSWER_KEY_DOCX).
# The STUDENT answers no longer pass through DOCX; they come straight from the
# OCR section (see the Bridge cell). normalize_ocr_lines() is already defined there.
try:
    import docx
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"]); import docx

from docx.document import Document as _DocClass
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import _Cell, Table
from docx.text.paragraph import Paragraph

def _iter_block_items(parent):
    """Yield Paragraphs and Tables in document order (preserves layout)."""
    if isinstance(parent, _DocClass):
        parent_elm = parent.element.body
    elif isinstance(parent, _Cell):
        parent_elm = parent._tc
    else:
        raise ValueError("unsupported parent")
    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

def load_docx_lines(path):
    """Return an ordered list of text 'lines' from a .docx: paragraphs split on
    soft line breaks; tables flattened row-by-row as 'cell | cell'."""
    doc = docx.Document(path)
    lines = []
    for block in _iter_block_items(doc):
        if isinstance(block, Paragraph):
            for sub in block.text.split("\n"):
                sub = sub.strip()
                if sub:
                    lines.append(sub)
        elif isinstance(block, Table):
            for row in block.rows:
                cells = [c.text.strip() for c in row.cells]
                row_txt = " | ".join([c for c in cells if c])
                if row_txt:
                    lines.append(row_txt)
    return normalize_ocr_lines(lines)

print("Bridge + DOCX reader ready.")

### 1.6b  Diagram-aware text source (v8.3) — keeps sketch descriptions + the handwriting around them, in reading order
Fixes *“text written after a diagram is not evaluated.”* The grader text is now rebuilt from the same `parse_layout` blocks the review HTML uses, so a sketch's description and the prose around it both reach grading. Also stops dropping image-embedded descriptions.

In [ ]:
# ============================================================================
#  v8.3  DIAGRAM-AWARE TEXT SOURCE  (inserted after the 1.6 bridge)
#  ---------------------------------------------------------------------------
#  Two fixes for "text written around a diagram is not evaluated":
#
#   (1) Image-alt safety. Chandra sometimes emits a sketch as a markdown image
#       ![<description>](data:image...). The old _strip_md_inline DROPPED the
#       whole thing — description included. We now KEEP the description text.
#
#   (2) Layout reading-order source. Instead of relying only on parse_markdown
#       (which can collapse a diagram page to an image placeholder), we rebuild
#       each page from the SAME parse_layout blocks the review HTML uses, IN
#       READING ORDER: text/table/math blocks -> their text; Figure/Diagram
#       blocks -> Chandra's description of the sketch. So the words a student
#       labelled on a sketch AND the prose they wrote under it both reach the
#       grader, positioned correctly relative to the (a)/(b)/(c) markers.
#
#  The student answers no longer silently lose their diagram-adjacent text.
# ============================================================================
import re as _re8

# (1) keep the image DESCRIPTION (alt text) instead of discarding it ----------
_MD_IMG = _re8.compile(r"!\[([^\]]*)\]\([^)]*\)")     # capture alt text
def _strip_md_inline(s):
    s = _MD_IMG.sub(lambda m: " " + (m.group(1) or "") + " ", s)  # ![desc](url) -> desc
    s = _MD_LINK.sub(r"\1", s)              # [text](url) -> text
    s = _EMPH_RE.sub("", s)                 # **bold** / *italic* / `code`
    s = _HTML_INLINE.sub(" ", s)            # <br>, <b>, ...
    return s

# (2) layout reading-order text source ---------------------------------------
TEXT_SOURCE      = "layout"   # "layout" (recommended) | "markdown" (old behaviour)
TEXT_MERGE_MD    = True       # also fold in any substantial markdown line the layout missed
_BLOCK_BREAK = _re8.compile(r"</(?:p|div|li|tr|h[1-6])>|<br\s*/?>", _re8.I)

def _block_text_to_lines(content):
    """parse_layout block content (may carry <p>/<br>/<math> tags) -> text lines,
    preserving line breaks so leading (a)/(b)/(c) markers survive for segmentation."""
    s = _BLOCK_BREAK.sub("\n", content or "")
    try:    s = _to_mathjax(s)              # <math>/<chem> -> inline \(..\)  (from 1.5c)
    except Exception: pass
    s = _re8.sub(r"<[^>]+>", " ", s)        # strip any remaining tags
    return [ln.strip() for ln in s.split("\n")]

def _layout_lines_for_page(raw, img):
    """One page -> lines from parse_layout blocks in reading order. Figure/Diagram
    blocks contribute Chandra's description. Returns None if layout is unavailable."""
    try:
        blocks = parse_layout(raw, img)     # parse_layout + VISUAL_LABELS from 1.5c
    except Exception:
        return None
    if not blocks:
        return None
    out = []
    for b in blocks:
        content = b.content or ""
        if (b.label or "") in VISUAL_LABELS:
            desc = _re8.sub(r"\s+", " ", _re8.sub(r"<[^>]+>", " ", content)).strip()
            if desc:
                out.append(desc)            # the sketch's description, as its own line
        else:
            out.extend(_block_text_to_lines(content))
    return [x for x in out if x]

def lines_from_ocr(raws, imgs):
    """Raw OCR + page images -> normalized evaluator lines. Per page: prefer the
    layout reconstruction (reading order + diagram text), fall back to markdown
    when layout is empty/unavailable, and (TEXT_MERGE_MD) re-add any substantial
    markdown line the layout dropped — maximising recall of the student's words."""
    use_layout = (globals().get("TEXT_SOURCE", "layout") == "layout")
    merge_md   = bool(globals().get("TEXT_MERGE_MD", True))
    raw_lines = []
    for raw, img in zip(raws, imgs):
        md  = markdown_to_lines(parse_markdown(raw))
        lay = _layout_lines_for_page(raw, img) if use_layout else None
        if lay is None:
            chosen = md
        else:
            chosen = list(lay)
            if merge_md:
                # v8.7: anchored interleave. Layout and markdown are both in reading
                # order; we keep layout (diagram-aware) and splice in ONLY the markdown
                # lines layout genuinely missed -- each placed right AFTER the layout line
                # it follows in markdown order, never dumped at the page end. The match key
                # strips leading list numbers ("1." / "1)" / "(a)"), so a markdown-renumbered
                # duplicate like "1. 1) Chemical..." is recognised as the existing
                # "1) Chemical..." and dropped -> kills the part-(b)->(c) bleed. Markers are
                # never spliced in.
                import re as _re_mm
                def _strip_enum(x):
                    x = x.strip(); prev = None
                    while x != prev:
                        prev = x
                        x = _re_mm.sub(r'^\s*[\(\[]?\s*(?:\d{1,3}|[a-hA-H]|[ivxIVX]{1,4})\s*[\)\].:]\s*', '', x)
                    return x
                _is_mk = lambda s: bool((globals().get("_detect_partmark") or (lambda x: None))(s)
                                        or (globals().get("_detect_qmark") or (lambda x: None))(s))
                _key = lambda s: _re_mm.sub(r"[^a-z0-9]", "", _strip_enum(s).lower())
                lay_keys = [_key(x) for x in chosen]
                pos = {}
                for _i, _k in enumerate(lay_keys):
                    pos.setdefault(_k, _i)
                have = set(lay_keys); ins = {}; anchor = -1
                for ln in md:
                    k = _key(ln)
                    if k in pos:
                        anchor = max(anchor, pos[k]); continue
                    if len(ln.split()) < 3 or not k or k in have or _is_mk(ln):
                        continue
                    ins.setdefault(anchor, []).append(ln); have.add(k)
                if ins:
                    rebuilt = list(ins.get(-1, []))
                    for _i, _line in enumerate(chosen):
                        rebuilt.append(_line); rebuilt.extend(ins.get(_i, []))
                    chosen = rebuilt
        raw_lines.extend(chosen)
    return normalize_ocr_lines(raw_lines)

print(f"Diagram-aware text source ready (TEXT_SOURCE={TEXT_SOURCE!r}, merge_md={TEXT_MERGE_MD}).")


### 1.7 Grading configuration (v7.5 entailment-driven + LLM blend)

In [ ]:
import re, unicodedata
import numpy as np
# ---- mapping encoder (Tier-2) ----
ENCODER_ID = "BAAI/bge-large-en-v1.5"
BGE_QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
MAP_DENSE_W, MAP_LEXICAL_W, MAP_POS_PRIOR, MAP_SIM_FLOOR = 0.70, 0.30, 0.05, 0.30
# ---- grading models ----
STS_MODEL_ID = "cross-encoder/stsb-roberta-large"
NLI_MODEL_ID = "roberta-large-mnli"
# ---- blend / rounding ----
W_RUBRIC, W_SEM, ROUND_STEP = 0.4, 0.6, 0.50   # portal defaults (semantic-leaning)
# ---- firing / gating (from v7.5 portal) ----
STS_FIRE, ENTAIL_FIRE, LEX_COVER = 0.50, 0.60, 0.60
KEYWORD_FUZZ, CONTRA_THRESHOLD = 88, 0.55
SEM_FLOOR, SEM_CEIL, REL_FLOOR = 0.40, 0.72, 0.25
PERFECT_STS, PERFECT_CONTRA = 0.95, 0.15
SEM_RECALL_BETA, SHORT_POINT_TOK = 2.0, 4
LEX_SEM_MIN = 0.45
STS_STRONG  = 0.82
ENTAIL_WEAK = 0.30
ENT_FLOOR, ENT_CEIL = 0.15, 0.60
MCQ_MAX_WORDS, MCQ_AMBIGUOUS_MULTI = 6, True
assert abs(W_RUBRIC + W_SEM - 1.0) < 1e-6
print(f"Grader: {STS_MODEL_ID} + {NLI_MODEL_ID} + {LLM_MODEL_ID} | blend {W_RUBRIC}/{W_SEM} | round {ROUND_STEP}")


### 1.8 Text helpers + auto-rubric (splits each model answer into key points for **partial credit within a part's marks**)

In [ ]:
from rapidfuzz import fuzz

_STOP_WORDS = set((
    "a an the is are was were be been being am of to in into on for and or but with "
    "as by it its this that these those at from up down out over under again further "
    "then once here there all any both each few more most other some such no nor not "
    "only own same so than too very can will just should now we you they he she i me "
    "my your our their them his her do does did doing have has had having which who "
    "whom whose what when where why how also etc using used use uses e g").split())

def _norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _compact(s):
    return re.sub(r"[\s\-_]", "", _norm(s))

def _content_tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", _norm(s)) if w not in _STOP_WORDS and len(w) > 2]

def _kw_hit(ans_norm, kw, ans_compact=None):
    k = _norm(kw)
    if not k: return False
    if k in ans_norm: return True
    if ans_compact is not None and _compact(kw) and _compact(kw) in ans_compact: return True
    if len(ans_norm) >= 4 and len(k) >= 4:
        return fuzz.partial_ratio(k, ans_norm) >= KEYWORD_FUZZ
    return False

# ---- splitters ----
def _split_sentences(text):
    return [p.strip() for p in re.split(r"(?:[.;\n?!]+)", text) if p.strip()]

def _split_list(text):
    tmp = re.sub(r"\b(?:and|&)\b", ",", text, flags=re.I)
    return [p.strip() for p in re.split(r"[,;\n]+", tmp) if p.strip()]

_CLAUSE_MARKERS = re.compile(
    r"\s*(?:,|;|\band\b|\bbut\b|\bwhich\b|\bthat\b|\bwho\b|\bwhere\b|\bwhen\b|"
    r"\bbecause\b|\bwhile\b|\balthough\b|\bthough\b|\bwhereas\b|\bso that\b)\s*", re.I)

def _split_clauses(sentence):
    parts = [p.strip() for p in _CLAUSE_MARKERS.split(sentence) if p and p.strip()]
    merged = []
    for p in parts:
        if len(_content_tokens(p)) < 2:
            if merged: merged[-1] += " " + p
            else: merged.append(p)
        elif merged and len(_content_tokens(merged[-1])) < 2:
            merged[-1] += " " + p
        else:
            merged.append(p)
    return [m for m in merged if m.strip()] or [sentence]

def _units(text):
    text = re.sub(r"\s+", " ", (text or "").strip())
    if not text: return []
    units = []
    for s in _split_sentences(text):
        units.extend(_split_clauses(s) if len(s.split()) > 12 else [s])
    units = [u for u in units if _content_tokens(u)]
    if text not in units: units.append(text)
    return units or [text]

# ---- auto-rubric (v4): classify the model answer, split marks across key points ----
_OPT_LEAD = re.compile(r"^\s*\(?([a-d])\)\s*[-.:)]?\s*", re.I)   # "(c) NameNode"

MCQ_MAX_WORDS = 6   # a real MCQ key is just an option; longer => subjective answer
def detect_mcq_option(model_ans):
    a = (model_ans or "").strip()
    m = _OPT_LEAD.match(a)
    if not m:
        return None
    if len(a[m.end():].split()) > MCQ_MAX_WORDS:   # leading "(c)" on a paragraph != MCQ
        return None
    return m.group(1).lower()

def classify_answer(correct):
    text = re.sub(r"\s+", " ", (correct or "").strip())
    if not text: return "objective", []
    wc = len(text.split())
    sents = _split_sentences(text)
    list_parts = _split_list(text)
    short_items = [p for p in list_parts if 0 < len(p.split()) <= 5]
    if wc <= MCQ_MAX_WORDS and len(sents) <= 1 and len(list_parts) <= 1:
        return "objective", [text]
    # enumeration only when CLEARLY a list: 4+ comma items, mostly short, or an explicit list cue
    listy = bool(re.search(r"(?:include|are|following|such as)\s*:?", text, re.I)) or text.count(",") >= 3
    if len(list_parts) >= 4 and len(short_items) >= max(4, int(0.7 * len(list_parts))) and listy:
        return "enumeration", list_parts
    points = []
    for s in sents:
        points.extend(_split_clauses(s) if len(s.split()) > 12 else [s])
    points = [p for p in points if _content_tokens(p)]
    if len(points) > 8: points = points[:8]
    return "conceptual", (points or [text])

def build_auto_rubric(model_ans, max_marks):
    """Model answer -> (qtype, [{'text','marks','keywords'}...])."""
    qtype, points = classify_answer(model_ans)
    n = max(1, len(points))
    base = round(max_marks / n, 4)
    rubric, alloc = [], 0.0
    for i, p in enumerate(points):
        m = round(max_marks - alloc, 4) if i == n - 1 else base
        alloc += m
        rubric.append({"text": p, "marks": m, "keywords": _content_tokens(p)})
    return qtype, rubric

print("Auto-rubric builder ready.")

### 1.8b Sub-part splitter (OCR-robust) — splits a question's text into (a)/(b)/(c)
After Tier-1 isolates a question blob, this splits it into parts. Tolerant of garbled
markers: `(a)`, `a)`, `a.`, `(a`, glued `Q1)a)`. Falls back to one whole part.

In [ ]:
import re
from collections import OrderedDict
# a part marker at the start of a line: optional '(', a single a-h letter, then ) or .
_PART_OPEN = re.compile(r'^\s*\(?\s*([a-hA-H])\s*[\).:]\s+')
_PART_GLUED = re.compile(r'^\s*\(?\s*([a-hA-H])\s*[\).:]\s*')   # even with no trailing space

def split_into_parts(text, expected=None):
    """Split a question blob into {'a': '...', 'b': '...'} on sub-part markers.
    `expected` (list like ['a','b','c'] from the question paper) is used only to
    validate/log; the split itself is driven by the markers in the text.
    Returns OrderedDict; {'_': text} when no parts are found."""
    if not text or not text.strip():
        return OrderedDict()
    lines = [l for l in text.split("\n") if l.strip()]
    parts, cur, buf = OrderedDict(), None, []
    def flush():
        if cur is not None and buf:
            parts[cur] = parts.get(cur, "") + (" " if parts.get(cur) else "") + " ".join(buf).strip()
    for ln in lines:
        m = _PART_OPEN.match(ln) or _PART_GLUED.match(ln)
        if m:
            flush(); cur = m.group(1).lower(); buf = [ln[m.end():].strip()]
        else:
            buf.append(ln.strip())
    flush()
    if not parts:                       # no markers found at all
        return OrderedDict([("_", " ".join(lines).strip())])
    if expected:
        miss = [p for p in expected if p not in parts]
        if miss: print(f"    [parts] expected {expected}, missing {miss} -> will positional/semantic-fill")
    return parts
print("Sub-part splitter ready.")


### 1.8c Question-paper marks extractor — per question **and per part** from `[N]` brackets
Pairs each `[N]` with the part on the **same OCR row** (Chandra gives line boxes), then
carries the question number down the page. Text fallback for flat OCR.

In [ ]:
import re
_Q_HEADER = re.compile(r"^\s*(?:Q\.?\s*)?(\d{1,2})(?=\s*[\.\)]|\s+\(|\s*$)\s*[\.\)]?\s*(?:\(([a-eA-E])\))?")
_PART_RE  = re.compile(r"\(([a-eA-E])\)")
_MARKS_RE = re.compile(r"\[\s*(\d{1,2})\s*\]")

def extract_qp_marks_from_lines(rows):
    """rows: list of dicts {'y':float,'text':str} (one visual row each).
    Returns {'Q1': {'a':8,'b':4,'c':4}, ...}."""
    rows = sorted(rows, key=lambda r: r["y"])
    rubric, cur_q = {}, None
    last_part = {}
    for row in rows:
        txt = row["text"]
        mq = _Q_HEADER.match(txt)
        if mq and mq.group(1):
            cur_q = f"Q{int(mq.group(1))}"; rubric.setdefault(cur_q, {})
        mm = _MARKS_RE.search(txt)
        if not mm:
            pm = _PART_RE.search(txt)
            if pm and cur_q: last_part[cur_q] = pm.group(1).lower()
            continue
        n = int(mm.group(1))
        pm = _PART_RE.search(txt)
        if pm: part = pm.group(1).lower()
        else:  part = last_part.get(cur_q)
        if cur_q is None or part is None:
            continue
        rubric[cur_q][part] = n; last_part[cur_q] = part
    return rubric

def extract_qp_marks_from_text(text):
    """Flat-text fallback (fragile on multi-column OCR — prefer the row version)."""
    rubric, cur_q, cur_part = {}, None, None
    tok = re.compile(r"(?:^|\n)\s*(\d{1,2})(?=\s*[\.\)]|\s+\()|\(([a-eA-E])\)|\[\s*(\d{1,2})\s*\]")
    for m in tok.finditer(text):
        q, p, mk = m.group(1), m.group(2), m.group(3)
        if q:    cur_q = f"Q{int(q)}"; cur_part = None; rubric.setdefault(cur_q, {})
        elif p:  cur_part = p.lower()
        elif mk and cur_q and cur_part is not None: rubric[cur_q][cur_part] = int(mk)
    return rubric

def rows_from_raws(raws, imgs):
    """Chandra raw layout -> visual rows {'y','text'} via parse_layout bboxes."""
    rows = []
    for raw, img in zip(raws, imgs):
        try:
            for b in parse_layout(raw, img):
                if b.content and b.bbox:
                    rows.append({"y": float(b.bbox[1]),
                                 "text": re.sub(r"<[^>]+>"," ", b.content)})
        except Exception:
            pass
    return rows

def qp_to_qids(qp_marks):
    """{'Q1':{'a':8,'b':4}} -> (QP_MARKS qid->mark, QP_PARTS q->[parts])."""
    QP_MARKS, QP_PARTS = {}, {}
    for q, parts in sorted(qp_marks.items(), key=lambda kv: int(kv[0][1:])):
        if set(parts) == {"_"} or not parts:
            QP_MARKS[q] = float(parts.get("_", 1.0)); QP_PARTS[q] = ["_"]
        else:
            QP_PARTS[q] = sorted(parts)
            for p, mk in parts.items():
                QP_MARKS[f"{q}{p}"] = float(mk)
    return QP_MARKS, QP_PARTS
print("Question-paper marks extractor ready.")


### 1.9 Tier 1 — OCR-robust marker-based question detection (v7)

In [ ]:
# ============================================================================
#  TIER 1  —  section-aware, multi-format marker detection   (v7: OCR-robust)
# ----------------------------------------------------------------------------
#  Handles every marker style seen across the real sheets:
#    "Ans - 1 :"   "Q1." + "Ans..."   "Q1 Ans .."   "Answer1 :-"   "Ans1 (c)"
#    "[Answer No-1] -"   "Ques-1." / "Ans-1."   "A1)/B11)/C16)"   "Q1 c)"
#    "Ans. (Q.1) .."   bare "1." "2." ...
#
#  v7 fix — OCR-robust answer-boundary detection
#  --------------------------------------------------------------------------
#  Handwriting OCR routinely MANGLES the question number on the marker line,
#  e.g. it reads "Ques1" as the bare word "Quest" (the '1' fuses into a 't'
#  stroke), or "Quesb" for "Ques6". The old detector needed a *readable* number,
#  so every such marker was missed -> the marker word leaked into the answer and
#  NO boundary was created -> all answers collapsed into one blob (Q1 ate Q1-Q6;
#  the rest reported "no answer detected").
#
#  Now a line that merely *starts* with a marker keyword
#      Q / Que / Ques / Quest / Question / Ans / Answer / Sol / Solution ...
#  is treated as an answer boundary EVEN WHEN the number is garbled or absent:
#     * if a number is still readable  -> Tier-1 hard label  (qid known)
#     * if the number is lost          -> a clean unlabelled chunk handed to
#                                         Tier-2, which decides *which* question
#                                         it is by matching against the key.
#  The marker word itself is never leaked into the answer text. Prose lines that
#  merely begin with such a word ("Questions include ...") are NOT split, because
#  a real marker is either alone, number-bearing, or separator/option-led.
# ============================================================================
_DIGIT_FIX = str.maketrans({"S":"5","s":"5","l":"1","I":"1","O":"0","o":"0","B":"8","Z":"2","z":"2","g":"9"})
_LABEL = r"(?:questions?|ques|que|answers?|answr|aswer|answ|ans|ams|solution|soln|sol|q)"   # longest-first
_NUM   = r"([0-9SslIOoBZzg]{1,3})"

# ---- section headers (consume; set current section) ----
_SEC_HDR = re.compile(r"^\s*\[?\s*(?:section|part)\s*[-:.]*\s*([abc])\b", re.IGNORECASE)
_SEC_ANS = re.compile(r"^\s*(?:answer|ans)\s*:?\s*[-\u2192]\s*([abc])\s*[.\]]?\s*$", re.IGNORECASE)
def _detect_section(line):
    m = _SEC_HDR.search(line) or _SEC_ANS.match(line)
    if m: return m.group(1).upper()
    low = line.lower()
    if re.search(r"\b(?:multiple\s+choice|mcq)\b", low): return "A"
    if re.search(r"\bvery\s+short\s+answer\b", low):     return "B"
    if re.search(r"\bshort\s+answer\b", low):            return "C"
    return None

# ---- markers ----
# labelled:  [opt '['] label [opt 'No'/'Number'] [sep] [opt '('] number [opt ')'/']']
_LABELED = re.compile(
    rf"^\s*\[?\s*(?P<lab>{_LABEL})\s*\.?\s*(?:no\.?|number)?\s*[-.:)\u2192]?\s*"
    rf"\(?\s*{_NUM}\s*[\)\]]?(?=[\s.:)\-\u2192\]]|$)\s*[.:)\-\u2192]?\s*",
    re.IGNORECASE)
# "Ans. (Q.1) ..."  -> answer label then an embedded (Q.N)
_LABELED_QPAREN = re.compile(
    rf"^\s*(?:answers?|ans|ams)\b[\s.:)\-\u2192]*\(?\s*q[\s.]*{_NUM}\s*[\)\]]?\s*[-.:)]?\s*",
    re.IGNORECASE)
# section-letter prefix:  "A1)"  "B11)"  "C16."
_SECLETTER = re.compile(rf"^\s*\[?\s*(?P<sec>[ABC])\s*{_NUM}\s*[\)\].:]\s*", re.IGNORECASE)
# bare number:  "1."  "(2)"  "3)"
_BARE = re.compile(rf"^\s*\(?\s*{_NUM}\s*[\)\]]?(?=[\s.):\u2192]|$)\s*[.):\u2192]\s+")
# leading "Ans/Answer/Sol" prefix on an answer body -> stripped as noise
_ANS_PREFIX = re.compile(rf"^\s*(?:answers?|ans|ams|solution|soln|sol)\b\s*(?:no\.?|number)?\s*[-.:)\u2192]?\s*", re.IGNORECASE)
def _strip_ans(t): return _ANS_PREFIX.sub("", t, count=1).strip()

# ---- v7: OCR-robust header (fires even when the number is garbled/missing) ----
# keyword (longest-first) + up to 3 OCR-garble letters; number is optional and,
# when present, must be followed by a boundary so it can't swallow a content letter.
_HDR_KW  = r"(?:questions?|quest|ques|que|answers?|answr|aswer|answ|ans|ams|solutions?|soln|sol)"
_HDR_NUM = r"(?P<num>[0-9SslIOoBZzg]{1,3})"
_HDR_RE  = re.compile(
    rf"^\s*\[?\s*(?P<lab>{_HDR_KW})[a-z]{{0,3}}"
    rf"\s*\.?\s*(?:no\.?|number)?\s*"
    rf"(?P<sep1>[-.:)\u2192\]]?)\s*"
    rf"\(?\s*(?:{_HDR_NUM}(?=[\s.:)\-\u2192\]]|$))?\s*(?P<close>[\)\]]?)"
    rf"\s*(?P<sep2>[-.:)\u2192]?)",
    re.IGNORECASE)
_OPT_AFTER = re.compile(r"^\s*\(?[a-dA-D][\).]")     # remaining body looks like an MCQ option "a)" "(b)" "c."

def _is_qlabel(lab):   # a QUESTION marker (prompt) rather than an ANSWER marker
    return lab.lower().startswith(("question", "quest", "ques", "que"))

def _clean_qnum(tok):
    f = (tok or "").translate(_DIGIT_FIX)
    return int(f) if f.isdigit() else None

_N_QUESTIONS = 200          # upper bound while parsing the KEY; tightened after key load
_QTYPE = {}                 # qid -> "objective"/"enumeration"/"conceptual" (set after key load)

def _map_qnum(n, section):
    """Generalised: trust any plausible global question number (the uploaded key
    defines which numbers exist; section headers are consumed but don't renumber)."""
    if n is None: return None
    if 1 <= n <= _N_QUESTIONS: return n
    return None

def _is_mcq(qid):
    return _QTYPE.get(qid) == "objective"

def classify_header(line):
    """v7 robust marker classifier for a line that starts with a marker keyword.
    Returns one of:
        ('hard', n, body, kind)  -> readable question number n  (Tier-1 label)
        ('soft', body)           -> marker but number garbled/missing (Tier-2)
        None                     -> not a marker (ordinary content / prose)
    `body` is the text remaining AFTER the marker (marker word never leaks)."""
    if " | " in line:                                  # flattened table row -> never a marker
        return None
    m = _HDR_RE.match(line)
    if not m:
        return None
    body = line[m.end():].strip()
    n    = _map_qnum(_clean_qnum(m.group("num")), None)
    kind = "Q" if _is_qlabel(m.group("lab")) else "A"
    if n is not None:
        return ("hard", n, body, kind)                 # number recovered -> hard label
    # number lost -> only treat as a marker when the line really looks like a header:
    sep = (m.group("sep1") or "") + (m.group("sep2") or "")
    if body == "":                                     # lone marker  e.g. "Quest" / "Ans:"
        return ("soft", "")
    if sep.strip():                                    # keyword + separator + content  e.g. "Ans- foo"
        return ("soft", body)
    if _OPT_AFTER.match(body):                          # keyword + MCQ option  e.g. "Quest a) ..."
        return ("soft", body)
    return None                                        # "Questions include ..." -> prose, not a marker

def _find_marker(line, max_seen, section):
    """Numbered formats: "Q1." / "Ans 1" / "A1)" / "1." / "Ans.(Q.1)".
    classify_header() runs first for garbled multi-letter markers; this preserves
    all the original numbered-marker detection exactly.
    -> (global_n, body_start_idx, kind, sec_hint).  kind: 'Q' prompt / 'A' answer."""
    if " | " in line:                                  # flattened table row -> never a marker
        return None, None, None, None
    m = _LABELED.match(line)
    if m:
        n = _map_qnum(_clean_qnum(m.group(2)), section)
        if n is not None:
            return n, m.end(), ("Q" if _is_qlabel(m.group("lab")) else "A"), None
    m = _LABELED_QPAREN.match(line)
    if m:
        n = _map_qnum(_clean_qnum(m.group(1)), section)
        if n is not None: return n, m.end(), "A", None
    m = _SECLETTER.match(line)
    if m:
        sec = m.group("sec").upper()
        n = _map_qnum(_clean_qnum(m.group(2)), sec)
        if n is not None: return n, m.end(), "A", sec
    m = _BARE.match(line)
    if m:
        n = _map_qnum(_clean_qnum(m.group(1)), section)
        if n is not None and n > max_seen:             # bare number only if it moves forward
            return n, m.end(), "A", None
    return None, None, None, None

def tier1_parse(lines):
    segments, cur_qid, buf, max_seen, section = [], None, [], 0, None

    def _flush():
        if buf or cur_qid is not None:
            segments.append((cur_qid, list(buf)))

    def _open_label(n, kind, rest):
        nonlocal cur_qid, buf, max_seen
        _flush()
        cur_qid  = f"Q{n}"
        max_seen = max(max_seen, n)
        buf      = [rest] if (rest and kind == "A") else []

    for ln in lines:
        sec = _detect_section(ln)
        if sec is not None:
            section = sec
            continue                                   # consume section header

        # 1) OCR-robust keyword header (the v7 fix) --------------------------
        h = classify_header(ln)
        if h is not None and h[0] == "hard":
            _, n, body, kind = h
            qid_new = f"Q{n}"
            rest = _strip_ans(body)
            if kind == "Q" and qid_new == cur_qid:
                continue                               # 'Question N' prompt for the open question -> ignore
            if kind == "A" and qid_new == cur_qid and not "".join(buf).strip():
                if rest: buf.append(rest)              # 'Answer N' right after its own 'Question N'
                continue
            _open_label(n, kind, rest)
            continue
        if h is not None and h[0] == "soft":
            body = h[1]                                # marker, number unreadable -> boundary for Tier-2
            if buf or cur_qid is not None:
                _flush()
            cur_qid = None
            buf = [body] if body else []
            continue

        # 2) other numbered marker formats  (A1) / 1. / Ans.(Q.1)) ----------
        n, end, kind, sec_hint = _find_marker(ln, max_seen, section)
        if sec_hint:
            section = sec_hint
        if n is not None:
            qid_new = f"Q{n}"
            rest = _strip_ans(ln[end:].strip())
            if kind == "Q" and qid_new == cur_qid:
                continue
            if kind == "A" and qid_new == cur_qid and not "".join(buf).strip():
                if rest: buf.append(rest)
                continue
            _open_label(n, kind, rest)
            continue

        # 3) ordinary content line ------------------------------------------
        if cur_qid is not None and _is_mcq(cur_qid) and "".join(buf).strip():
            continue                                   # MCQ answer already captured -> drop trailing options
        buf.append(ln)

    _flush()

    # ---- dedup: any qid claimed twice is ambiguous -> demote both to Tier 2 ----
    seen, dup = {}, set()
    for qid, _ in segments:
        if qid is None: continue
        if qid in seen: dup.add(qid)
        seen[qid] = seen.get(qid, 0) + 1
    mapping, leftovers = {}, []
    for qid, b in segments:
        text = "\n".join(b).strip()
        if qid is None:
            if text: leftovers.append(text)
        elif qid in dup:
            if text: leftovers.append(text)
        else:
            mapping[qid] = text
    if dup: print(f"[Tier1] demoted conflicting qids to Tier 2: {sorted(dup)}")
    return mapping, leftovers

### 1.10 Load encoders — BGE (mapping) + STS + NLI (grading; lazy)

In [ ]:
# ---------------- BGE bi-encoder (Tier-2 question<->answer MAPPING only) ----------------
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(ENCODER_ID)
print("Mapping encoder:", ENCODER_ID, "| dim:", encoder.get_sentence_embedding_dimension())

from functools import lru_cache

def _encode(texts, is_query=False):
    if is_query:
        texts = [BGE_QUERY_INSTRUCTION + t for t in texts]
    return encoder.encode(list(texts), normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=False)

@lru_cache(maxsize=4096)
def _emb_one(text, is_query=False):
    return _encode([text], is_query=is_query)[0]

# ---------------- v4 grading models: STS cross-encoder + NLI (lazy-loaded) ----------------
import torch as _torch
_STS = None
_NLI = None

def get_sts():
    global _STS
    if _STS is None:
        from sentence_transformers import CrossEncoder
        _STS = CrossEncoder(STS_MODEL_ID,
                            device="cuda" if _torch.cuda.is_available() else "cpu")
        print("Loaded STS cross-encoder:", STS_MODEL_ID)
    return _STS

def get_nli():
    global _NLI
    if _NLI is None:
        from transformers import pipeline
        _NLI = pipeline("text-classification", model=NLI_MODEL_ID, top_k=None,
                        device=0 if _torch.cuda.is_available() else -1)
        print("Loaded NLI model:", NLI_MODEL_ID)
    return _NLI

def _sts(pairs):
    """STS similarity in [0,1] for a list of (text_a, text_b) pairs."""
    if not pairs:
        return []
    scores = get_sts().predict(pairs, batch_size=32, show_progress_bar=False)
    return [float(max(0.0, min(1.0, s))) for s in np.atleast_1d(scores)]

def _nli(pairs):
    """NLI label probabilities for (premise, hypothesis) pairs."""
    if not pairs:
        return []
    inputs = [{"text": p, "text_pair": h} for (p, h) in pairs]
    out = get_nli()(inputs, batch_size=16, truncation=True)
    if out and isinstance(out[0], dict):
        out = [out]
    res = []
    for row in out:
        d = {"contradiction": 0.0, "entailment": 0.0, "neutral": 0.0}
        for item in row:
            lab = str(item["label"]).lower()
            if "contradict" in lab:
                d["contradiction"] = float(item["score"])
            elif "entail" in lab:
                d["entailment"] = float(item["score"])
            elif "neutral" in lab:
                d["neutral"] = float(item["score"])
        res.append(d)
    return res


### 1.10b Qwen3-8B grader (4-bit) — leads grading; STS/NLI flags disagreements
Notebook-native port of the portal's hybrid: when LLM and engine agree within
`LLM_DISAGREE` they blend (LLM weighted `LLM_WEIGHT`); when they disagree the LLM is
trusted and the answer is flagged `review`. Lazy-loaded, 4-bit nf4 so it shares the GPU.

In [ ]:
import threading
_LLM = {"model": None, "tok": None, "ready": False}
_LLM_LOCK = threading.Lock()
_LLM_SYS = ("You are a fair exam grader for HANDWRITTEN answers transcribed by OCR, so expect "
            "spelling slips, garbled words and stray characters -- judge the MEANING, not the spelling. "
            "Award marks in PROPORTION to how many of the model answer's key points the student covers; "
            "covering most key points earns most of the marks. A clear paraphrase, or an obvious "
            "misspelling / OCR-garble of the right term (e.g. 'hydroxazine' for 'hydrazine'), is CORRECT. "
            "Give near-zero ONLY when the answer is missing, off-topic, or states something genuinely "
            "WRONG -- never zero an otherwise-correct answer over a typo or a few stray words. Ignore "
            "stray text that clearly belongs to another question. Reply with JSON only, no other text.")

def get_llm():
    if _LLM["ready"]:
        return _LLM["model"], _LLM["tok"]
    with _LLM_LOCK:
        if _LLM["ready"]:
            return _LLM["model"], _LLM["tok"]
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        tok = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
        kw = {"trust_remote_code": True}
        if torch.cuda.is_available() and LLM_4BIT:
            try:
                from transformers import BitsAndBytesConfig
                kw.update(quantization_config=BitsAndBytesConfig(
                            load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.float16),
                          device_map={"": 0})
            except Exception:
                kw.update(torch_dtype=torch.float16, device_map={"": 0})
        elif torch.cuda.is_available():
            kw.update(torch_dtype=torch.float16, device_map={"": 0})
        else:
            kw.update(torch_dtype=torch.float32)
        m = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **kw); m.eval()
        _LLM.update(model=m, tok=tok, ready=True)
        print("Loaded LLM grader:", LLM_MODEL_ID, "(4-bit)" if LLM_4BIT else "")
        return m, tok

def _llm_chat(system, user, max_new_tokens=160):
    """Single Qwen turn -> decoded text. Handles tensor-vs-BatchEncoding return shapes."""
    import torch
    m, tok = get_llm()
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    def _tmpl(**extra):
        return tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True, **extra)
    try:
        enc = _tmpl(enable_thinking=False)
    except TypeError:
        try: enc = _tmpl()
        except TypeError:
            enc = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                          return_tensors="pt")
    if isinstance(enc, dict) or hasattr(enc, "input_ids"):
        enc = {k: v.to(m.device) for k, v in dict(enc).items()}
        input_len = enc["input_ids"].shape[1]; gen_args, gen_kwargs = (), enc
    else:
        ids = enc.to(m.device); input_len = ids.shape[1]; gen_args, gen_kwargs = (ids,), {}
    with torch.no_grad():
        out = m.generate(*gen_args, **gen_kwargs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=(tok.eos_token_id or tok.pad_token_id))
    text = tok.decode(out[0][input_len:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S)

def llm_grade(model_ans, student, maxm, question=""):
    """(fraction 0..1, reason) from Qwen, or None on failure. Reads the batched
    cache filled by prime_llm_grades() so the per-part loop is instant."""
    if not LLM_ENABLED:
        return None
    _k = _grade_key((model_ans or "").strip(), (student or "").strip(), float(maxm), question or "")
    if _k in _GRADE_CACHE:
        return _GRADE_CACHE[_k]
    try:
        import json
        qline = (f"Exam question:\n{question.strip()}\n\n" if (question or "").strip() else "")
        user = (qline +
                f"Model (correct) answer:\n{model_ans}\n\n"
                f"Student answer:\n{(student or '').strip() or '(blank)'}\n\n"
                f"Maximum marks: {maxm}\n\n"
                f"Award marks for how well the student answer addresses THIS question and "
                f"matches the meaning of the model answer. A blank, off-topic, or wrong-key-term "
                f"answer scores near zero.\n"
                f'Reply ONLY with JSON: {{"score": <number from 0 to {maxm}>, '
                f'"reason": "<one short sentence>"}}.')
        text = _llm_chat(_LLM_SYS, user, max_new_tokens=96)   # was 160 — faster on T4
        mm = re.search(r"\{.*\}", text, re.S)
        if not mm: return None
        data = json.loads(mm.group(0))
        score = max(0.0, min(float(data.get("score")), float(maxm)))
        return score / max(float(maxm), 1e-9), str(data.get("reason", ""))[:200]
    except Exception:
        import traceback; traceback.print_exc()
        return None

_LLM_MAP_SYS = ("You assign each answer segment from an exam answer booklet to the question it "
                "answers. Read the CONTENT carefully — different questions can share keywords "
                "(e.g. two questions may both mention lubricating oil), so decide by what the "
                "answer is actually about. Candidates may answer questions in ANY order and may "
                "skip questions, so do NOT assume the numbers increase. Reply with JSON only.")
def llm_map_questions(seg_texts, paper_q_text, qorder, label="ans"):
    """One Qwen call: map each answer segment -> question id. Returns list aligned to
    seg_texts (qid or None). None on failure (caller falls back to the heuristic)."""
    if not LLM_ENABLED or not seg_texts:
        return None
    try:
        import json
        qlist = "\n".join(f"{q}: {paper_q_text.get(q,'')[:170]}" for q in qorder)
        seglist = "\n".join(f"[{i}] {(' '.join(t.split()))[:380]}" for i, t in enumerate(seg_texts))
        user = (f"QUESTIONS:\n{qlist}\n\n"
                f"ANSWER SEGMENTS (in the order written, {len(seg_texts)} of them):\n{seglist}\n\n"
                f"For every segment index 0..{len(seg_texts)-1}, output which question it answers "
                f"(one of: {', '.join(qorder)}). Answers may be in ANY order; assign each by its "
                f'CONTENT. Reply ONLY with JSON mapping index to qid, e.g. {{"0":"Q2","1":"Q2","2":"Q9"}}.')
        text = _llm_chat(_LLM_MAP_SYS, user, max_new_tokens=40 + 12*len(seg_texts))
        mm = re.search(r"\{.*\}", text, re.S)
        if not mm: return None
        raw = json.loads(mm.group(0))
        valid = set(qorder)
        out = []
        for i in range(len(seg_texts)):
            v = raw.get(str(i)) or raw.get(i)
            v = (str(v).strip().upper().replace(" ", "") if v is not None else None)
            if v and not v.startswith("Q"): v = "Q" + v
            out.append(v if v in valid else None)
        return out
    except Exception:
        import traceback; traceback.print_exc()
        return None
print("Qwen grader + LLM mapper wired (lazy).")

# ===========================================================================
#  v8.5  BATCHED Qwen grading — same model, ~5x faster on a T4
#  The per-part loop streamed the 8B weights once PER PART (~26x). Batching
#  streams them once per BATCH. Inputs are capped and outputs kept short.
# ===========================================================================
import hashlib
_GRADE_CACHE = {}                       # content-key -> (frac, reason) | None

def _grade_key(model_ans, student, maxm, question):
    h = f"{question}||{model_ans}||{student}||{maxm}".encode("utf-8", "ignore")
    return hashlib.md5(h).hexdigest()

def _cap(s, n=600):
    s = (s or "").strip()
    return s if len(s) <= n else s[:n] + " …"

def _build_grade_prompt(model_ans, student, maxm, question):
    qline = (f"QUESTION:\n{_cap(question,300)}\n\n" if (question or '').strip() else "")
    stu = _cap(student)
    stu_block = stu if stu else "[the student left this part empty]"
    return (qline +
            f"REFERENCE ANSWER (worth full marks):\n{_cap(model_ans)}\n\n"
            f"STUDENT ANSWER:\n{stu_block}\n\n"
            f"Grade the STUDENT ANSWER out of {maxm} by how many of the reference answer's KEY POINTS it "
            f"covers, giving PROPORTIONAL partial credit (covers 3 of 4 key points -> about 3/4 of the marks). "
            f"Treat misspelled or OCR-garbled forms of the correct term as correct, and IGNORE stray words, "
            f"stray characters, or text that clearly belongs to another question. Do NOT give marks for merely "
            f"repeating the question wording or vague filler, and do NOT zero an otherwise-correct answer "
            f"because of a typo or a single off-topic phrase. "
            f"Write the reason from what the student ACTUALLY wrote (quote a phrase if useful); "
            f"do not call an answer empty unless the STUDENT ANSWER block literally says it is empty.\n"
            f'Reply with compact JSON only: {{"score": <0..{maxm}>, "reason": "<=12 words, concrete>"}}.')

def _parse_grade(text, maxm):
    import json
    mm = re.search(r"\{.*?\}", text, re.S)
    if not mm: return None
    try:
        data  = json.loads(mm.group(0))
        score = max(0.0, min(float(data.get("score")), float(maxm)))
        return score / max(float(maxm), 1e-9), str(data.get("reason", ""))[:200]
    except Exception:
        return None

def _llm_chat_batch(systems, users, max_new_tokens=64):
    """Left-padded batched generation -> list[str]. Honors LLM_BATCH_SIZE."""
    import torch
    m, tok = get_llm()
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    bs, outs = max(1, int(globals().get("LLM_BATCH_SIZE", 4))), []
    for i in range(0, len(users), bs):
        prompts = []
        for sysmsg, usr in zip(systems[i:i+bs], users[i:i+bs]):
            msgs = [{"role": "system", "content": sysmsg}, {"role": "user", "content": usr}]
            try:
                p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                            enable_thinking=False)
            except TypeError:
                p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            prompts.append(p)
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(m.device)
        with torch.no_grad():
            gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=(tok.pad_token_id or tok.eos_token_id))
        start = enc["input_ids"].shape[1]
        for j in range(gen.shape[0]):
            t = tok.decode(gen[j][start:], skip_special_tokens=True)
            outs.append(re.sub(r"<think>.*?</think>", "", t, flags=re.S))
    return outs

def prime_llm_grades(qids):
    """Batch-grade every (non-MCQ) part once and cache it, so the per-part
    grade_question() calls below are instant. Call this right before the loop."""
    if not LLM_ENABLED:
        return
    items = []
    for qid in qids:
        if (AUTO_RUBRIC.get(qid, {}) or {}).get("qtype") == "objective":
            continue                                       # MCQ path doesn't use the LLM
        student   = (answer_map.get(qid, "") or "").strip()
        model_ans = (MODEL_ANSWERS.get(qid, "") or "").strip()
        maxm      = float(QUESTION_MARKS[qid])
        qtext     = question_for(qid) if "question_for" in globals() else ""
        items.append((model_ans, student, maxm, qtext))
    if not items:
        return
    import time
    t0 = time.time()
    print(f"  [LLM] batch-grading {len(items)} parts "
          f"(batch_size={globals().get('LLM_BATCH_SIZE',4)}, ~{LLM_GRADE_TOKENS} tok each)…")
    texts = _llm_chat_batch([_LLM_SYS]*len(items),
                            [_build_grade_prompt(*it) for it in items],
                            max_new_tokens=int(globals().get("LLM_GRADE_TOKENS", 64)))
    for (ma, st, mm, qt), text in zip(items, texts):
        _GRADE_CACHE[_grade_key(ma, st, mm, qt)] = _parse_grade(text, mm)
    # v8.6: retry the handful whose batched JSON did not parse, one-by-one with more
    # room, so they are graded by the LLM instead of silently falling back to the engine.
    retry = [it for it in items if _GRADE_CACHE.get(_grade_key(*it)) is None]
    if retry:
        print(f"  [LLM] retrying {len(retry)} unparsed part(s) individually…")
        for (ma, st, mm, qt) in retry:
            try:
                txt = _llm_chat(_LLM_SYS, _build_grade_prompt(ma, st, mm, qt), max_new_tokens=128)
                _GRADE_CACHE[_grade_key(ma, st, mm, qt)] = _parse_grade(txt, mm)
            except Exception:
                pass
    print(f"  [LLM] grading finished in {time.time()-t0:.0f}s")


### 1.11 Tier 2 — Hungarian alignment (now also used to align student parts within a question)

In [ ]:
import re as _re
from scipy.optimize import linear_sum_assignment

_STOP = set("a an the is are of to in on for and or with as by it its this that be can has have not".split())
def _tokens(s):
    return {w for w in _re.findall(r"[a-z0-9]+", s.lower()) if w not in _STOP and len(w) > 1}

def _lexical_overlap(a, b):
    ta, tb = _tokens(a), _tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)            # Jaccard

def _ref_text(qid):
    """Reference text for mapping = the model answer + its auto-rubric key points."""
    parts = [MODEL_ANSWERS.get(qid, "")]
    for p in AUTO_RUBRIC.get(qid, {}).get("points", []):
        parts.append(p["text"])
    return " ; ".join(p for p in parts if p)

def tier2_align(chunks, unfilled_qids):
    if not chunks or not unfilled_qids:
        return {}
    refs = [_ref_text(q) for q in unfilled_qids]
    Ec = _encode(chunks, is_query=True)           # chunks are the "queries"
    Er = _encode(refs)
    dense = Ec @ Er.T                              # (n_chunks, n_q) cosine
    M, N = dense.shape
    S = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            lex = _lexical_overlap(chunks[i], refs[j])
            pos = 1.0 - abs(i/max(M-1,1) - j/max(N-1,1))
            S[i, j] = MAP_DENSE_W*dense[i, j] + MAP_LEXICAL_W*lex + MAP_POS_PRIOR*pos
    rows, colspad = linear_sum_assignment(-S)      # maximise total similarity
    out = {}
    for r, c in zip(rows, colspad):
        if S[r, c] >= MAP_SIM_FLOOR:
            out[unfilled_qids[c]] = chunks[r]
    return out

# ---- v8: align a student's part-chunks to the key's parts WITHIN one question ----
def align_parts(student_parts, key_parts):
    """student_parts/key_parts: OrderedDict {part_letter: text}.
    Returns {key_part_letter: student_text}. Positional when the letters line up;
    otherwise BGE + Hungarian on the part texts (reuses the Tier-2 machinery)."""
    if not key_parts:
        return {}
    if not student_parts:
        return {kp: "" for kp in key_parts}
    sk, kk = list(student_parts.keys()), list(key_parts.keys())
    # exact letter alignment when the student used the same labels
    if set(kk).issubset(set(sk)) or set(sk) == set(kk):
        return {kp: student_parts.get(kp, "") for kp in kk}
    # student parts unlabelled ('_') or mismatched -> semantic assignment
    s_txt = [student_parts[s] for s in sk]
    k_txt = [key_parts[k] for k in kk]
    Es = _encode(s_txt, is_query=True); Ek = _encode(k_txt)
    dense = Es @ Ek.T
    from scipy.optimize import linear_sum_assignment
    rows, cols = linear_sum_assignment(-dense)
    out = {kk[c]: s_txt[r] for r, c in zip(rows, cols)}
    for k in kk: out.setdefault(k, "")
    return out

# ---- v8: parts-aware reference text. The lifted tier2 _ref_text looks up
# MODEL_ANSWERS[qid]; in parts mode answers are stored as Q1a/Q1b, so a question-id
# like "Q1" must aggregate its parts for the Tier-2 leftover rescue to match. ----
def _ref_text(qid):
    if qid in MODEL_ANSWERS:                       # a part-id (Q1a) — use it directly
        parts = [MODEL_ANSWERS.get(qid, "")]
        for p in AUTO_RUBRIC.get(qid, {}).get("points", []):
            parts.append(p["text"])
        return " ; ".join(x for x in parts if x)
    agg = [MODEL_ANSWERS[k] for k in MODEL_ANSWERS  # a question-id (Q1) — aggregate parts
           if re.match(rf"{re.escape(qid)}[a-h_]?$", k)]
    return " ; ".join(x for x in agg if x)
print("Tier-2 + part-alignment ready.")


### 1.11b Combined question+part parser & `map_parts` — handles `Q1. (a)` (key) and `Q1)a)` (student)
The model key repeats the question number on every part (`Q1. (a)`, `Q1. (b)`), which the
question-prompt logic in Tier-1 would skip. This parser reads question **and** part boundaries
directly. `map_parts` uses it first, then falls back to Tier-1 blob + `split_into_parts` +
`align_parts` for any expected part it didn't find — so a one-blob answer is still recovered.

In [ ]:
import re
_QLAB = r"(?:q(?:ues(?:tion)?)?|ans(?:wer)?|sol(?:ution)?)"
_NUMC = r"[0-9SslIOoBZzg]"
# combined marker: optional q/ans label, optional question number, REQUIRED part letter.
# part may sit at end of line ("Q1. (b)") or lead content ("(a) Following ...").
_QPART_RE = re.compile(
    rf"^\s*\[?\s*(?:{_QLAB}\s*\.?\s*(?:no\.?)?\s*)?"
    rf"(?:\(?\s*({_NUMC}{{1,3}})\s*[\)\].:]?\s*)?"
    rf"\(?\s*([a-hA-H])\s*[\)\.:](?:\s+|\s*$)", re.I)
# question-only marker (no part): "Q1." "Q2)" "1." "Ans 3 -"
_QONLY_RE = re.compile(
    rf"^\s*\[?\s*(?:{_QLAB}\s*\.?\s*(?:no\.?)?\s*)?"
    rf"\(?\s*({_NUMC}{{1,3}})\s*[\)\].:](?:\s+|\s*$)", re.I)

def parse_qparts(lines, qp_parts):
    """Direct question+part parse -> {'Q1a': text, ...}. Robust to the key's
    repeated 'Q1. (a)' markers and the student's 'Q1)a)' / bare '(b)'."""
    out, cur_q, cur_part = {}, None, None
    def _key():
        if cur_q is None: return None
        if cur_part is None:
            return cur_q if qp_parts.get(cur_q, ["_"]) == ["_"] else None
        return f"{cur_q}{cur_part}"
    for ln in lines:
        if " | " in ln:                     # flattened table row -> never a marker
            k = _key()
            if k: out.setdefault(k, []).append(ln.strip())
            continue
        mp = _QPART_RE.match(ln)
        if mp:
            if mp.group(1):
                n = _clean_qnum(mp.group(1))
                if n is not None: cur_q = f"Q{n}"
            cur_part = mp.group(2).lower()
            body = ln[mp.end():].strip()
            k = _key()
            if k is not None and body: out.setdefault(k, []).append(body)
            continue
        mo = _QONLY_RE.match(ln)
        if mo:
            n = _clean_qnum(mo.group(1))
            if n is not None:
                cur_q = f"Q{n}"; cur_part = None
                body = ln[mo.end():].strip()
                k = _key()
                if k is not None and body: out.setdefault(k, []).append(body)
            continue
        k = _key()
        if k is not None: out.setdefault(k, []).append(ln.strip())
    return {k: " ".join(v).strip() for k, v in out.items() if " ".join(v).strip()}

def map_parts(lines, qp_parts, key_parts=None):
    """Map OCR lines -> {qid: answer_text} for every expected qid in qp_parts.
    Primary = parse_qparts; fallback = Tier-1 blob -> split_into_parts -> align_parts."""
    primary = parse_qparts(lines, qp_parts)
    qblobs, leftovers = tier1_parse(lines)
    qbases = list(qp_parts.keys())
    unfilled_q = [q for q in qbases
                  if q not in qblobs and not any(k.startswith(q) for k in primary)]
    if leftovers and unfilled_q:
        qblobs = {**tier2_align(leftovers, unfilled_q), **qblobs}
    out = {}
    for q in qbases:
        parts = qp_parts[q]
        if parts == ["_"]:
            out[q] = primary.get(q) or qblobs.get(q, ""); continue
        have = {p: primary.get(f"{q}{p}", "") for p in parts}
        if not any(have.values()):                          # parse found no parts -> split blob
            blob = qblobs.get(q, "")
            sp = split_into_parts(blob, expected=parts)
            if key_parts and q in key_parts:
                aligned = align_parts(sp, key_parts[q])
                have = {p: aligned.get(p, "") for p in parts}
            else:
                vals = list(sp.values()) if (sp and list(sp)[0] != "_") else [blob]
                have = {p: (vals[i] if i < len(vals) else "") for i, p in enumerate(parts)}
        for p in parts:
            out[f"{q}{p}"] = have.get(p, "")
    return out

# ---- v8.2: LLM-LED mapping. OCR garbles the key's handwritten question numbers
# (Q3->"A18", Q4->"Q1" dup, Q9->"Q."/"89.") AND look-alike topics fool BGE (Q2 lube-oil
# conveyance vs Q8 lube-oil piping). So: detect the part marker (robust to junk), then let
# QWEN read each answer + the questions and assign the question. A deterministic
# sequence+number heuristic is the fallback when the LLM is off/unavailable. ----
_PARTMARK = re.compile(r"^\s*[^\n]{0,5}?\(\s*([a-hA-H])\s*\)|^\s*\(?\s*([a-hA-H])\s*[\)\.:]\s+", re.I)
def _detect_partmark(ln):
    m = _PARTMARK.match(ln)
    if not m: return None
    g = 1 if m.group(1) else 2
    letter = m.group(g).lower()
    pre = ln[:m.start(g)].translate(_DIGIT_FIX)
    mn = re.search(r"\d{1,2}", pre)
    num = int(mn.group(0)) if mn else None
    if num is not None and not (1 <= num <= 20): num = None
    return num, letter, m.end()

def _segment_by_parts(lines):
    """Split OCR lines into answer segments at each detected part marker."""
    segs, cur = [], None
    for ln in lines:
        d = _detect_partmark(ln)
        if d is not None:
            num, letter, end = d
            cur = {"num": num, "letter": letter, "lines": [ln[end:].strip()]}
            segs.append(cur)
        elif cur is not None:
            cur["lines"].append(ln.strip())
    return segs

def _seq_question_for_segments(segs, qorder, qp_parts):
    """Deterministic fallback: assign each segment a question using the booklet order.
    Part letter 'a' (or a letter <= the previous one) starts a NEW question; b/c continue.
    A valid, non-decreasing OCR number is trusted; otherwise the sequence decides."""
    out, cur_qi, prev_letter = [], -1, None
    for s in segs:
        letter = s["letter"]
        num_qi = qorder.index(f"Q{s['num']}") if (s["num"] and f"Q{s['num']}" in qp_parts) else None
        if prev_letter is not None and letter > prev_letter:      # b after a, c after b -> same q
            qi = cur_qi if cur_qi >= 0 else 0
            if num_qi is not None and num_qi > cur_qi: qi = num_qi
        else:                                                      # reset -> new question
            if num_qi is not None and num_qi >= cur_qi + 1: qi = num_qi      # valid forward number
            else: qi = cur_qi + 1                                            # garbled/dup -> next
        qi = max(0, min(qi, len(qorder) - 1))
        if qi < cur_qi: qi = cur_qi
        out.append(qorder[qi]); cur_qi = qi; prev_letter = letter
    return out

def sbert_map_questions(seg_texts, paper_q_text, qorder, label="ans"):
    """SBERT mapping: score every (question, answer-segment) pair with the STS CROSS-encoder
    (reads both texts together, so it separates look-alike topics like Q2 vs Q8 that a
    bi-encoder blurs), then choose a NON-DECREASING assignment via DP (answers are in
    booklet order). A gentle diagonal prior breaks ties toward the expected position.
    Returns a qid per segment, or None on failure (caller falls back)."""
    if not seg_texts or not paper_q_text:
        return None
    try:
        import numpy as np
        n, m = len(seg_texts), len(qorder)
        refs = [paper_q_text.get(q, "") for q in qorder]
        pairs = [(refs[j], (seg_texts[i] or "")[:500]) for i in range(n) for j in range(m)]
        scores = np.asarray(_sts(pairs), dtype=float).reshape(n, m)     # cross-encoder
        # gentle pull toward the diagonal (segment i is expected around question i*m/n)
        pos = np.linspace(0, m - 1, n)
        for i in range(n):
            scores[i] -= 0.04 * np.abs(np.arange(m) - pos[i])
        # monotonic DP: each segment -> a question, questions non-decreasing
        NEG = -1e9
        dp = np.full((n, m), NEG); back = np.zeros((n, m), dtype=int)
        dp[0] = scores[0]; back[0] = np.arange(m)
        for i in range(1, n):
            best_prev, best_j = NEG, 0
            for j in range(m):
                if dp[i-1][j] > best_prev: best_prev, best_j = dp[i-1][j], j
                dp[i][j] = scores[i][j] + best_prev; back[i][j] = best_j
        j = int(np.argmax(dp[n-1])); assign = [0]*n
        for i in range(n-1, -1, -1):
            assign[i] = j; j = back[i][j]
        return [qorder[a] for a in assign]
    except Exception:
        import traceback; traceback.print_exc()
        return None

def map_parts_smart(lines, qp_parts, paper_q_text, label="ans", verbose=True):
    """Segment by part markers, then assign each segment's QUESTION using MAP_METHOD
    ('sbert' cross-encoder + monotonic DP, 'llm' Qwen, or 'sequence' heuristic). The part
    comes from the detected letter. Always falls back to the deterministic sequence map."""
    qorder = sorted(qp_parts, key=lambda q: int(q[1:]))
    segs = _segment_by_parts(lines)
    if not segs:
        if verbose: print(f"  [map:{label}] no part markers found — using label parser")
        return map_parts(lines, qp_parts)
    seg_txt = [" ".join(s["lines"]).strip() for s in segs]
    seq_q = _seq_question_for_segments(segs, qorder, qp_parts)

    method = globals().get("MAP_METHOD", "sbert")
    model_q = None
    if paper_q_text:
        if method == "sbert":   model_q = sbert_map_questions(seg_txt, paper_q_text, qorder, label)
        elif method == "llm":   model_q = llm_map_questions(seg_txt, paper_q_text, qorder, label)
    how = method if model_q is not None else "sequence"
    if model_q is None:
        if verbose and method != "sequence":
            print(f"  [map:{label}] {method} mapping unavailable — using sequence heuristic")
        qassign = seq_q
    else:
        qassign, cur_qi = [], -1
        for i, s in enumerate(segs):
            q = model_q[i] or seq_q[i]
            qi = qorder.index(q)
            if qi < cur_qi: qi = cur_qi                      # enforce booklet order
            q = qorder[qi]; cur_qi = qi
            if verbose and model_q[i] and model_q[i] != seq_q[i]:
                print(f"  [map:{label}] seg{i} (part {s['letter']}): {method}->{model_q[i]}  seq->{seq_q[i]}  using {q}")
            qassign.append(q)

    out_lines = {}
    for s, txt, q in zip(segs, seg_txt, qassign):
        parts = qp_parts[q]; letter = s["letter"]
        qid = q if parts == ["_"] else f"{q}{letter if letter in parts else parts[0]}"
        out_lines.setdefault(qid, []).append(txt)
    out = {qid: " ".join(v).strip() for qid, v in out_lines.items()}
    for q in qorder:
        for p in qp_parts[q]:
            out.setdefault(q if p == "_" else f"{q}{p}", "")
    if verbose:
        filled = sum(1 for v in out.values() if v.strip())
        print(f"  [map:{label}] {filled}/{len(out)} parts filled ({how})")
    return out

# keep the old name as an alias so existing cells keep working
map_parts_anchored = map_parts_smart
print(f"Mapper ready — method = '{MAP_METHOD}' (sbert cross-encoder / llm / sequence).")


### 1.11c  Order-independent, content-anchored mapper (v8.3) — **the main fix**
Replaces the booklet-order (monotonic) assumption. Candidates answer any 6 of 9 questions in **any** order; each answer block is matched to its question by **content against both the question text and the model answer**, then blocks are assigned to questions **globally (Hungarian)** so each question is used once. Re-defines `map_parts_anchored`, so Parts 2–3 below pick it up automatically.

In [ ]:
import re
from collections import OrderedDict
import numpy as np
_QLAB = r"(?:q(?:ues(?:tion)?)?|ans(?:wer)?|sol(?:ution)?)"
_NUMC = r"[0-9SslIOoBZzg]"

# ===========================================================================
#  v8.4  ROBUST, ORDER-INDEPENDENT, CONTENT-ANCHORED MAPPING
#  Fixes the v8.3 regression: the thin _PARTMARK missed the dominant real
#  marker style "(2) a)" (number in parens + part) while catching "(3) (a)",
#  which scrambled the whole mapping. This restores the robust combined
#  marker detection (handles "(2) a)", "(3) (a)", "Q1. (a)", "1(a)",
#  "Ans1 (c)", "Q1 c)", bare "a)") AND recovers OCR-garbled marker shells
#  ("(1) (1)", "U) 161", "Q)(1)", "(9) 101"), while still rejecting the
#  enumerated list items ("1)", "2)") students write inside a part.
# ===========================================================================
try:
    _DIGIT_FIX
except NameError:
    _DIGIT_FIX = str.maketrans({"S":"5","s":"5","l":"1","I":"1","O":"0","o":"0",
                                "B":"8","Z":"2","z":"2","g":"9"})

def _qnum_from(text):
    if not text: return None
    t = text.translate(_DIGIT_FIX)
    m = re.search(r"\d{1,2}", t)
    if not m: return None
    n = int(m.group(0))
    return n if 1 <= n <= 20 else None

# combined marker: optional Q/Ans label, optional (N) (delimiter optional), REQUIRED letter
_QPART_RE = re.compile(
    rf"^\s*\[?\s*(?:{_QLAB}\s*[-.:]?\s*(?:no\.?)?\s*)?"
    rf"(?:\(?\s*({_NUMC}{{1,2}})\s*[\)\].:]?\s*)?"
    rf"\(?\s*([a-hA-H])\s*[\)\.:][\):.\-]*(?:\s+|\s*$)", re.I)
# question-only marker: a LABEL ('Q1','Ans 3') OR parens '(N)' — NOT a bare 'N)' (list item)
_QONLY_RE = re.compile(
    rf"^\s*\[?\s*(?:"
    rf"{_QLAB}\s*[-.:]?\s*(?:no\.?)?\s*\(?\s*({_NUMC}{{1,2}})\s*[\)\].:]?"
    rf"|\(\s*({_NUMC}{{1,2}})\s*\)"
    rf")(?:\s+|\W|$)", re.I)

def _detect_partmark(ln):
    m = _QPART_RE.match(ln)
    if not m: return None
    num = _qnum_from(m.group(1)) if m.group(1) else None
    return num, m.group(2).lower(), m.end()

def _detect_qmark(ln):
    if _QPART_RE.match(ln): return None
    m = _QONLY_RE.match(ln)
    if not m: return None
    num = _qnum_from(m.group(1) or m.group(2))
    return (num, m.end()) if num is not None else None

# page-footer / barcode-band noise that must never be treated as a marker.
# NB: only square-bracket '[6]' or bare '6' — a round '(6)' may be a real Q-marker.
_FOOTER_RE = re.compile(r"^\s*(?:\[\s*\d{1,3}\s*\]|\d{1,3})\s*$")
_BARCODE_RE = re.compile(r"^[\s|I1l]{6,}$")                          # OCR'd barcode bars
_PAGE_RE    = re.compile(r"^\s*page\s*\d{1,3}\s*$", re.I)                  # bridge 'Page N' separators
_PB_Q_RE    = re.compile(r"^\s*\(?\s*(\d{1,3})\s*"
                         r"(?:[\).]\s+(?=\S)"                       # 'NN) text' / 'NN. text'
                          r"|[\.\-:]?\s*\(\s*[a-zA-Z0-9]\s*\))")  # 'NN-(1)' / 'NN.(a)' / 'NN(a)' (garbled part)
_ENUM_LEAD_RE = re.compile(r"^\s*\(?\s*(\d{1,3})\s*[\).]")                  # running list counter

def _segment_by_parts(lines):
    """OCR lines -> answer segments at each part/question marker. A bare 'Q2)' line opens a
       NEW (part-less) segment that following unlabelled text fills, so a multi-page answer
       continuing with no marker is not swallowed by the previous part. 'Page N' separators
       and page footers/barcodes are dropped; right after a page break a leading bare
       'N)'/'N.' may be read as a question marker (handwritten 'Q4)' often OCRs as '24)')."""
    segs, cur, pending = [], None, None
    at_page_start, last_enum = True, None
    for ln in lines:
        if _FOOTER_RE.match(ln) or _BARCODE_RE.match(ln):
            continue
        if _PAGE_RE.match(ln):
            at_page_start = True; continue                 # page break -> note it, drop the line
        d = _detect_partmark(ln)
        if d is not None:
            num, letter, end = d
            if num is None: num, pending = pending, None
            cur = {"num": num, "letter": letter, "lines": [ln[end:].strip()]}
            segs.append(cur); at_page_start = False; last_enum = None; continue
        q = _detect_qmark(ln)
        if q is None and at_page_start:                    # bare 'N)' right after a page break
            mm = _PB_Q_RE.match(ln)
            if mm:
                n = int(mm.group(1))
                is_list_cont = (cur is not None and last_enum is not None and n == last_enum + 1)
                if not is_list_cont:                       # a real jump -> treat as a question marker
                    q = (n, mm.end())
        if q is not None:
            num, end = q; body = ln[end:].strip()
            if body:
                cur = {"num": num, "letter": None, "lines": [body]}; segs.append(cur); pending=None
            else:
                pending = num                              # bare 'Q2)' -> tag the NEXT text
            at_page_start = False; last_enum = None; continue
        # plain text line
        em = _ENUM_LEAD_RE.match(ln)
        if em: last_enum = int(em.group(1))
        if pending is not None:
            cur = {"num": pending, "letter": None, "lines": [ln.strip()]}   # open the pending question
            segs.append(cur); pending = None
        elif cur is not None:
            cur["lines"].append(ln.strip())
        at_page_start = False
    for s in segs:
        s["text"] = " ".join(x for x in s["lines"] if x).strip()
    return [s for s in segs if s["text"]]

# ---- group segments into question-blocks (number-primary, letter-secondary) ----
def _ord(letter): return ord(letter) - ord("a") if letter else -1

def _group_blocks(segs):
    blocks, cur = [], None
    prev_letter, cur_num = None, None
    for s in segs:
        letter, num = s["letter"], s["num"]
        new = False
        if cur is None:
            new = True
        elif num is not None and cur_num is not None and num != cur_num:
            new = True                                   # different question number
        elif letter is not None and prev_letter is not None and _ord(letter) <= _ord(prev_letter):
            new = True                                   # part-letter reset (numbers absent/equal)
        if new:
            cur = {"segs": [s], "nums": [num] if num else []}
            blocks.append(cur); cur_num = num; prev_letter = letter
        else:
            cur["segs"].append(s)
            if num: cur["nums"].append(num); cur_num = cur_num if cur_num is not None else num
            if letter is not None: prev_letter = letter
    for b in blocks:
        b["text"] = " ".join(s["text"] for s in b["segs"]).strip()
        b["num"]  = max(set(b["nums"]), key=b["nums"].count) if b["nums"] else None
    return _merge_consecutive_samenum(blocks)

def _first_letter(b):
    for s in b["segs"]:
        if s["letter"] is not None: return s["letter"]
    return None
def _last_letter(b):
    for s in reversed(b["segs"]):
        if s["letter"] is not None: return s["letter"]
    return None

def _merge_consecutive_samenum(blocks):
    """Re-stated or OCR-duplicated markers ('(3)(a)' twice across a page break, or the key's
       duplicate '(b)') split one question into several blocks. Merge a block into the
       previous one when (a) they carry the same written number, or (b) the later block has
       NO number and its first part-letter does not RESET below the previous block's last
       letter -- i.e. it is the same question's parts continuing, not a new question."""
    out = []
    for b in blocks:
        merge = False
        if out:
            same_num = (b["num"] is not None and out[-1]["num"] == b["num"])
            if same_num:
                merge = True
            elif b["num"] is None:                       # numberless continuation?
                pl, cl = _last_letter(out[-1]), _first_letter(b)
                if pl is None or cl is None or _ord(cl) >= _ord(pl):
                    merge = True
        if merge:
            out[-1]["segs"].extend(b["segs"]); out[-1]["nums"].extend(b["nums"])
            out[-1]["text"] = (out[-1]["text"] + " " + b["text"]).strip()
            if out[-1]["num"] is None and b["num"] is not None:
                out[-1]["num"] = b["num"];
        else:
            out.append(b)
    return out

# ---- reference text: question text (+ model answer for students) ----
def _question_text(q):
    fn = globals().get("question_for")
    if fn:
        try:
            t = fn(q)
            if t and t.strip(): return t
        except Exception: pass
    return (globals().get("PAPER_Q_TEXT", {}) or {}).get(q, "")

def _model_answer_for(q, qp_parts):
    MA = globals().get("MODEL_ANSWERS", {}) or {}
    parts = qp_parts.get(q, ["_"])
    if parts == ["_"]: return MA.get(q, "")
    return " ".join(MA.get(f"{q}{p}", "") for p in parts).strip()

def _sts_or_bge(pairs):
    if not pairs: return []
    if "_sts" in globals():
        try: return list(globals()["_sts"](pairs))
        except Exception: pass
    A = globals()["_encode"]([a for a, _ in pairs], is_query=True)
    B = globals()["_encode"]([b for _, b in pairs])
    return [float(np.dot(A[i], B[i])) for i in range(len(pairs))]

MAP_W_QTEXT, MAP_W_ANSWER, MAP_NUM_BONUS, MAP_TXT_CAP = 0.35, 0.65, 0.15, 600

def _block_question_scores(blocks, qorder, qp_parts, use_model):
    n, m = len(blocks), len(qorder)
    qtexts = [(_question_text(q) or "")[:MAP_TXT_CAP] for q in qorder]
    atexts = [(_model_answer_for(q, qp_parts) or "")[:MAP_TXT_CAP] for q in qorder] if use_model else [""]*m
    btexts = [(b["text"] or "")[:MAP_TXT_CAP] for b in blocks]
    q_pairs = [(btexts[i], qtexts[j]) for i in range(n) for j in range(m)]
    q_sc = np.asarray(_sts_or_bge(q_pairs), float).reshape(n, m) if q_pairs else np.zeros((n, m))
    if use_model and any(atexts):
        a_pairs = [(btexts[i], atexts[j]) for i in range(n) for j in range(m) if atexts[j]]
        a_it = iter(_sts_or_bge(a_pairs)); a_sc = np.zeros((n, m))
        for i in range(n):
            for j in range(m):
                a_sc[i, j] = next(a_it) if atexts[j] else 0.0
        has = np.array([bool(a) for a in atexts])
        S = q_sc * np.where(has, MAP_W_QTEXT, 1.0)[None,:] + a_sc * np.where(has, MAP_W_ANSWER, 0.0)[None,:]
    else:
        S = q_sc
    for i, b in enumerate(blocks):
        if b["num"] is not None and f"Q{b['num']}" in qorder:
            S[i, qorder.index(f"Q{b['num']}")] += MAP_NUM_BONUS
    return S

def _assign_blocks(S, qorder):
    """Assign each block to the question its content+number best matches,
       INDEPENDENTLY (not one-to-one). On these sheets a single question is
       routinely split across several blocks (multi-page answers, duplicated or
       OCR-garbled markers); a one-to-one assignment would scatter the extra
       blocks onto wrong questions. Blocks that pick the same question are merged
       afterwards. The +num bonus in S guards against genuine topic collapse."""
    n, _ = S.shape
    return [qorder[int(np.argmax(S[i]))] for i in range(n)]

# ---- distribute a block's parts to the question's slots (position-aware) ----
# ---- marker-driven question assignment: the WRITTEN number wins, never SBERT ----
def _assign_by_marker(blocks, qorder, qp_parts, use_model):
    """A block whose written number names a real, NOT-YET-SEEN question is assigned to it
    directly (the marker decides). A numberless block continues the previous question. A block
    whose number is garbled/out-of-range OR repeats an already-assigned question (a second
    'Q1' that is really a misread 'Q4') is resolved by CONTENT among the questions."""
    assign = [None] * len(blocks)
    seen, garbled, prev_q = set(), [], None
    for i, b in enumerate(blocks):
        n = b["num"]; q = f"Q{n}" if n is not None else None
        if q in qp_parts and q not in seen:
            assign[i] = q; seen.add(q); prev_q = q       # first sighting of this number -> trust it
        elif n is None and prev_q is not None:
            assign[i] = prev_q                           # numberless -> continue previous question
        else:
            garbled.append(i)                            # invalid OR duplicate number -> content
    if garbled:
        Ssub = _block_question_scores([blocks[i] for i in garbled], qorder, qp_parts, use_model)
        sub  = _assign_blocks(Ssub, qorder)
        for idx, q in zip(garbled, sub):
            assign[idx] = q
    by_mark = len(blocks) - len(garbled)
    return assign, f"marker ({by_mark}/{len(blocks)} by number, {len(garbled)} by content/dup)"

# ---- distribute a block's parts to the question's slots (label-driven) ----
def _distribute_parts(block, q, qp_parts):
    parts = qp_parts[q]
    if parts == ["_"]:
        return {q: block["text"]}
    by_letter, unknown = OrderedDict(), []
    for s in block["segs"]:
        L = s["letter"]
        if L in parts:
            by_letter.setdefault(L, []).append(s["text"])   # student wrote the letter -> trust it
        else:
            unknown.append(s["text"])                        # no/garbled letter -> place positionally
    # v8.6: a LABELLED segment stays in its own slot; SBERT/align never moves it.
    # Only unlabelled leftovers fill the still-empty key slots, in reading order.
    empty = [p for p in parts if p not in by_letter]
    for txt, p in zip(unknown, empty):
        by_letter.setdefault(p, []).append(txt)
    extra = unknown[len(empty):]
    if extra:
        last = list(by_letter.keys())[-1] if by_letter else parts[0]
        by_letter.setdefault(last, []).extend(extra)
    return {f"{q}{p}": " ".join(by_letter.get(p, [])).strip() for p in parts}

# ===========================================================================
def map_parts_smart(lines, qp_parts, paper_q_text=None, label="ans", verbose=True):
    qorder = sorted(qp_parts, key=lambda q: int(q[1:]))
    segs = _segment_by_parts(lines)
    if not segs:
        if verbose: print(f"  [map:{label}] no markers found - falling back to label parser")
        return map_parts(lines, qp_parts)
    use_model = (label != "key") and bool(globals().get("MODEL_ANSWERS"))
    method = globals().get("MAP_METHOD", "marker")
    if label == "key":
        # The key's OCR markers are badly garbled -- lost part-letters ('Q1.(a).'), wrong
        # numbers (Q4->'Q1', Q9->'89'), missing numbers (Q3), duplicate '(b)', and a later
        # question's text GLUED onto an earlier block. So for the key we align at the
        # SEGMENT level and map each segment to a question by CONTENT under a
        # non-decreasing-order constraint (the key is always written Q1->Q9). Finer units +
        # content + order are robust to all of the above; then parts distribute by letter.
        blocks = [{"segs": [s], "text": s["text"], "num": s["num"],
                   "nums": [s["num"]] if s["num"] else []} for s in segs]
        try:
            S = _block_question_scores(blocks, qorder, qp_parts, use_model=False)
            assign = _assign_monotonic_dp(S, qorder); how = "key: content+order (segment-level)"
        except Exception as _e:
            blocks = _group_blocks(segs)
            assign = _seq_monotonic(blocks, qorder, qp_parts); how = f"key: order only ({_e})"
    else:
        blocks = _group_blocks(segs)
        if method == "sequence":
            assign = _seq_fallback(blocks, qorder, qp_parts); how = "sequence"
        elif method == "marker":
            assign, how = _assign_by_marker(blocks, qorder, qp_parts, use_model)
        else:
            S = _block_question_scores(blocks, qorder, qp_parts, use_model)
            assign = _assign_blocks(S, qorder)
            how = "content+answer" if use_model else "content"
    merged = OrderedDict()
    for b, q in zip(blocks, assign): merged.setdefault(q, []).append(b)
    out = {}
    for q, bl in merged.items():
        if len(bl) == 1:
            part_map = _distribute_parts(bl[0], q, qp_parts)
        else:
            combo = {"segs": [s for b in bl for s in b["segs"]], "text": " ".join(b["text"] for b in bl)}
            part_map = _distribute_parts(combo, q, qp_parts)
        out.update(part_map)
    for q in qorder:
        for p in qp_parts[q]:
            out.setdefault(q if p == "_" else f"{q}{p}", "")
    if verbose:
        filled = sum(1 for v in out.values() if v.strip())
        print(f"  [map:{label}] {len(blocks)} block(s) -> {filled}/{len(out)} parts filled ({how})")
        for b, q in zip(blocks, assign):
            letters = "".join(s["letter"] or "_" for s in b["segs"])
            head = (b["text"][:54] + "…") if len(b["text"]) > 54 else b["text"]
            print(f"      block[{letters:<4} ocr#{str(b['num']):>3}] -> {q:<4} | {head}")
    return out

def _seq_fallback(blocks, qorder, qp_parts):
    out, qi = [], -1
    for b in blocks:
        n = b["num"]
        cand = qorder.index(f"Q{n}") if (n and f"Q{n}" in qp_parts) else qi + 1
        qi = max(0, min(cand, len(qorder) - 1)); out.append(qorder[qi])
    return out

def _assign_monotonic_dp(S, qorder):
    """Assign blocks to questions by content score S (n_blocks x n_q) under the constraint
    that the question index is NON-DECREASING down the page. The model-answer key is always
    written Q1->Q9 in order, so this tolerates garbled/duplicate/missing markers (e.g. 'Qs.',
    'Q.', '89.', duplicate '(b)') while never letting content pull a block out of order."""
    import numpy as np
    S = np.asarray(S, dtype=float)
    n, m = S.shape
    NEG = -1e9
    dp = np.full((n, m), NEG); bk = -np.ones((n, m), dtype=int)
    dp[0, :] = S[0, :]
    for i in range(1, n):
        pmax, parg = NEG, 0
        pm = np.empty(m); pa = np.empty(m, dtype=int)
        for j in range(m):                       # prefix max over previous row (q' <= j)
            if dp[i-1, j] > pmax: pmax, parg = dp[i-1, j], j
            pm[j], pa[j] = pmax, parg
        for j in range(m):
            dp[i, j] = S[i, j] + pm[j]; bk[i, j] = pa[j]
    j = int(np.argmax(dp[n-1, :])); out = [None]*n
    for i in range(n-1, -1, -1):
        out[i] = qorder[j]
        if bk[i, j] >= 0: j = int(bk[i, j])
    return out

def _seq_monotonic(blocks, qorder, qp_parts):
    """The model-answer KEY is written in question order, but its OCR numbers are often
    garbled (Q4->'Q1', Q9->'89', Q3 dropped). Ignore a number that would jump BACKWARDS
    and simply advance through the questions in document order."""
    out, qi = [], -1
    for b in blocks:
        n = b["num"]
        idx = qorder.index(f"Q{n}") if (n and f"Q{n}" in qp_parts) else None
        if idx is not None and idx >= qi:
            qi = idx                                  # number agrees / jumps forward -> take it
        else:
            qi = min(qi + 1, len(qorder) - 1)         # numberless or backwards -> next question
        out.append(qorder[qi])
    return out

map_parts_anchored = map_parts_smart
print(f"v8.8 mapper ready (student=marker-driven, key=content+order DP, multi-page stitch (markers: (N) a) / (N)(a) / Q1(a) / Ans1(c) / garbled shells; "
      f"order-independent; MAP_METHOD='{globals().get('MAP_METHOD','anchor')}').")


### 1.12 The grader — Qwen leads, STS/NLI cross-check & flag (v7.5 entailment engine)
Reads the globals built in Part 2 (`MODEL_ANSWERS`, `QUESTION_MARKS`, `AUTO_RUBRIC`, `answer_map`).
Per (question-)part: MCQ → lexical; else entailment-driven STS/NLI engine → LLM blend/flag.

In [ ]:
import math
_NLI_ZERO = {"contradiction": 0.0, "entailment": 0.0, "neutral": 0.0}
def _fbeta(p, r, beta=SEM_RECALL_BETA):
    b2 = beta*beta; d = b2*p + r
    return ((1+b2)*p*r/d) if d > 0 else 0.0
def _cal(x):     return max(0.0, min(1.0, (x - SEM_FLOOR) / (SEM_CEIL - SEM_FLOOR)))
def _cal_ent(x): return max(0.0, min(1.0, (x - ENT_FLOOR) / (ENT_CEIL - ENT_FLOOR)))
def _round_step(x, step=ROUND_STEP):
    return round(math.floor(x/step + 0.5 + 1e-9)*step, 2) if step else round(x, 2)

# MCQ scoring (unchanged from your engine)
_OPT_PAREN = re.compile(r"\(\s*([a-d])\s*\)", re.I)
_OPT_WORD  = re.compile(r"\b(?:option|opt|choice|ans(?:wer)?)\s*[-:.)]*\s*\(?([a-d])\)?(?![\w])", re.I)
_OPT_LONE  = re.compile(r"^\s*\(?([a-d])\)?\s*[.):\-]?\s*$", re.I)
def _detect_option_letters(ans):
    letters = {m.lower() for m in _OPT_PAREN.findall(ans)} | {m.lower() for m in _OPT_WORD.findall(ans)}
    m = _OPT_LONE.match(ans.strip())
    if m: letters.add(m.group(1).lower())
    return letters
def score_mcq(student, model_ans, maxm, option=None):
    if not student.strip(): return 0.0, "no answer detected"
    body = _OPT_LEAD.sub("", model_ans).strip() or model_ans
    an, ac = _norm(student), _compact(student)
    letters = _detect_option_letters(student)
    if MCQ_AMBIGUOUS_MULTI and len(letters) >= 2:
        return 0.0, f"ambiguous: multiple options {sorted(letters)}"
    lex = _kw_hit(an, body, ac) or any(_kw_hit(an, t, ac) for t in _content_tokens(body))
    if not lex:
        mnum = re.match(r"\s*(\d+)", body)
        if mnum and re.search(rf"\b{mnum.group(1)}\b", an): lex = True
    opt = bool(letters) and option and (letters == {option})
    if opt and not lex: return maxm, f"correct (option letter '{option}')"
    if lex:             return maxm, "correct (answer matched)"
    if letters and option:
        return 0.0, f"wrong option '{sorted(letters)[0]}' (correct: '{option}')"
    return 0.0, "wrong / not matched"

def grade_question(qid):
    entry  = AUTO_RUBRIC[qid]; qtype = entry["qtype"]; points = entry["points"]; option = entry.get("option")
    maxm   = float(QUESTION_MARKS[qid])
    student   = (answer_map.get(qid, "") or "").strip()
    model_ans = (MODEL_ANSWERS.get(qid, "") or "").strip()
    if qtype == "objective":
        marks, why = score_mcq(student, model_ans, maxm, option)
        base = {"qid": qid, "qtype": "objective", "max_marks": maxm, "marks": marks,
                "rubric_frac": marks/maxm if maxm else 0.0, "sem_credit": None,
                "final_frac": marks/maxm if maxm else 0.0, "why": why, "review": False,
                "llm_frac": None, "engine_frac": marks/maxm if maxm else 0.0, "llm_reason": "",
                "llm_marks": None, "engine_marks": marks, "decision": "MCQ (exact match)",
                "question": question_for(qid) if "question_for" in globals() else "",
                "perfect": marks >= maxm, "contradiction": False, "off_topic": False,
                "meaning": None, "sts_full": None, "contra_full": None, "points": [],
                "student": student, "model": model_ans}
        return base

    live  = bool(student)
    exact = bool(live and model_ans and _norm(student) == _norm(model_ans))
    s_units = _units(student) if live else []
    pts_txt = [p["text"] for p in points]
    if live and s_units:
        sts_full = _sts([(model_ans, student)])[0] if model_ans else 0.0
        nli_cs   = _nli([(model_ans, student)])[0] if model_ans else dict(_NLI_ZERO)
        nli_sc   = _nli([(student, model_ans)])[0] if model_ans else dict(_NLI_ZERO)
        flat = [(p, u) for p in pts_txt for u in s_units]
        M = _sts(flat); cols = len(s_units)
        kp_best   = [max(M[i*cols:(i+1)*cols]) for i in range(len(pts_txt))]
        unit_best = [max(M[i*cols+j] for i in range(len(pts_txt))) for j in range(cols)]
        kp_nli    = _nli([(student, p) for p in pts_txt])
    else:
        sts_full = 0.0; nli_cs = nli_sc = dict(_NLI_ZERO)
        kp_best = [0.0]*len(points); unit_best = []; kp_nli = [dict(_NLI_ZERO) for _ in points]

    contra_full = max(nli_cs["contradiction"], nli_sc["contradiction"])
    sn, sc = _norm(student), _compact(student)
    points_out, rub_marks = [], 0.0
    for k, p in enumerate(points):
        sim, ent, con = kp_best[k], kp_nli[k]["entailment"], kp_nli[k]["contradiction"]
        if not live:
            fired, contr, how = False, False, "missed"
        else:
            kws = p["keywords"]
            cover = (sum(1 for kw in kws if _kw_hit(sn, kw, sc)) / len(kws)) if kws else 0.0
            lex = cover >= LEX_COVER or _kw_hit(sn, p["text"], sc)
            contr = (con >= CONTRA_THRESHOLD)
            short = len(_content_tokens(p["text"])) <= SHORT_POINT_TOK
            if qtype == "enumeration":
                base = lex or (sim >= STS_FIRE); how_fire = "keyword" if lex else "semantic"
            else:
                entailed = ent >= ENTAIL_FIRE
                strong_para = (sim >= STS_STRONG) and (ent >= ENTAIL_WEAK)
                base = entailed or strong_para
                how_fire = "entailment" if entailed else "paraphrase"
            fired = base and not contr
            how = "contradicted" if contr else (how_fire if fired else "missed")
        awarded = p["marks"] if fired else 0.0
        rub_marks += awarded
        points_out.append({"text": p["text"], "marks": p["marks"], "fired": fired,
                           "contradicted": bool(contr and live), "how": how, "awarded": awarded,
                           "sim": round(sim,3), "entail": round(ent,3), "contra": round(con,3)})

    rub_marks = min(rub_marks, maxm); rub_frac = (rub_marks/maxm) if maxm else 0.0
    recall = float(np.mean(kp_best)) if kp_best else 0.0
    precision = float(np.mean(unit_best)) if unit_best else 0.0
    soft_f1 = _fbeta(precision, recall)
    entail_full = nli_sc["entailment"]
    sem_credit = _cal(soft_f1) * (1.0 - contra_full) * _cal_ent(entail_full)
    off_topic = bool(live and sts_full < REL_FLOOR and precision < 0.35)
    if off_topic: sem_credit = 0.0
    capped  = bool(live and contra_full >= CONTRA_THRESHOLD and rub_marks == 0)
    perfect = bool((exact or (live and sts_full >= PERFECT_STS and contra_full <= PERFECT_CONTRA))
                   and not capped and not off_topic)
    final_frac = W_RUBRIC*rub_frac + W_SEM*sem_credit
    final_frac = max(final_frac, W_RUBRIC*rub_frac)
    if perfect: final_frac = 1.0
    if capped or off_topic: final_frac = 0.0
    marks = min(_round_step(maxm*final_frac), maxm)
    fired_n = sum(1 for q in points_out if q["fired"])
    if not live:    why = "no answer detected"
    elif off_topic: why = "off-topic (unrelated to the question)"
    elif capped:    why = "contradiction: asserts the opposite of the model answer"
    elif perfect:   why = "correct / equivalent to the model answer"
    else:
        covered = [q["text"][:40] for q in points_out if q["fired"]]
        why = f"{fired_n}/{len(points_out)} key points" + ((": " + "; ".join(covered)) if covered else "")

    # ---- Qwen cross-check (agreement-aware), grading WITH the exam question ----
    review, llm_frac, llm_reason = False, None, ""
    engine_frac = final_frac
    qtext = question_for(qid) if "question_for" in globals() else ""
    lg = llm_grade(model_ans, student, maxm, question=qtext)
    if lg is not None:
        llm_frac, llm_reason = lg
        # v8.6: the LLM (Qwen) is the SOLE grader. The STS/NLI engine NEVER changes the
        # score — it only raises a [REVIEW] flag when it disagrees by > LLM_DISAGREE.
        final_frac = max(0.0, min(1.0, llm_frac))
        review = abs(llm_frac - engine_frac) > LLM_DISAGREE
        marks = min(_round_step(maxm*final_frac), maxm)
        tag = "[REVIEW] " if review else ""
        decision = "LLM only" + ("  ·  engine disagrees -> review" if review else "  ·  engine agrees")
        why = (f"{tag}LLM {round(llm_frac*maxm,2)}/{maxm}" + (f": {llm_reason}" if llm_reason else "")
               + f"  ·  engine {round(engine_frac*maxm,2)}/{maxm} (flag only)")
    else:
        # LLM genuinely unavailable for this part -> engine fallback, flagged for review.
        review = True
        decision = "engine fallback (LLM unavailable) -> review"

    llm_marks    = (round(llm_frac*maxm, 2) if llm_frac is not None else None)
    engine_marks = round(engine_frac*maxm, 2)
    return {"qid": qid, "qtype": qtype, "max_marks": maxm, "marks": marks,
            "rubric_frac": round(rub_frac,3), "sem_credit": round(sem_credit,3),
            "final_frac": round(final_frac,3), "why": why, "review": review,
            "llm_frac": (round(llm_frac,3) if llm_frac is not None else None),
            "engine_frac": round(engine_frac,3), "llm_reason": llm_reason,
            "llm_marks": llm_marks, "engine_marks": engine_marks, "decision": decision,
            "question": qtext,
            "perfect": perfect, "contradiction": capped, "off_topic": off_topic,
            "meaning": {"precision": round(precision,3), "recall": round(recall,3), "f_beta": round(soft_f1,3)},
            "sts_full": round(sts_full,3), "contra_full": round(contra_full,3),
            "points": points_out, "student": student, "model": model_ans}
print("Grader ready — Qwen leads, STS/NLI flags.")


### 1.12b  v8.9 fixes — empty-answer guard · prompt hardening · garbled-marker recovery · diagram embedding
Run once (overrides the relevant functions defined above; the mapper/grader pick them up automatically).

In [ ]:
# ===========================================================================
#  v8.9  FIXES  (drop-in overrides — must run once, after Part 1 is defined)
#  ---------------------------------------------------------------------------
#  1) EMPTY-ANSWER GUARD  -> a blank student part is ALWAYS 0 and is never sent
#     to the LLM, so the grader can no longer "grade" the model answer.
#  2) PROMPT HARDENING    -> the LLM is told to score ONLY the student's text and
#     never to credit a key point that lives only in the reference answer.
#  3) GARBLED-MARKER RECOVERY -> "02-(1)"-style markers (the (a) OCR'd as (1))
#     now open the right question, so diagram answers like Q2a stop bleeding
#     into the previous part.
#  4) DUPLICATE-MARKER ROUTING -> a misread duplicate number (Q4a OCR'd as Q1a)
#     is routed to the right question by CONTENT, with its wrong number's bonus
#     suppressed so it can't snap back onto the duplicated question.
#  5) DIAGRAMS IN REPORT  -> embed_student_diagrams() pulls every sketch crop
#     from the student OCR HTML into the DOCX (so the diagrams are visible).
# ===========================================================================
import re as _re89, base64 as _b64_89

# ---- shared helper: is a student part effectively empty? -------------------
_EMPTY_TOKENS = {"empty","(empty)","blank","(blank)","na","nil","none",
                 "noanswer","notattempted","-","—"}
def student_is_empty(text):
    if not text:
        return True
    t = str(text).strip().lower()
    if t.replace(" ", "") in _EMPTY_TOKENS:
        return True
    return len(_re89.sub(r"[^a-z0-9]+", "", t)) < 4   # only a stray marker/punct left

# ---- (2) hardened grading prompt (anti-leak) -------------------------------
def _build_grade_prompt(model_ans, student, maxm, question):
    qline = (f"QUESTION:\n{_cap(question,300)}\n\n" if (question or '').strip() else "")
    stu = _cap(student)
    stu_block = stu if stu else "[the student left this part empty]"
    return (qline +
            f"REFERENCE ANSWER (full-mark answer — for comparison ONLY, it is NOT the student's work):\n"
            f"{_cap(model_ans)}\n\n"
            f"STUDENT ANSWER:\n{stu_block}\n\n"
            f"Grade ONLY what appears in the STUDENT ANSWER block, out of {maxm}, by how many of the "
            f"reference answer's KEY POINTS the student actually wrote, with PROPORTIONAL partial credit "
            f"(covers 3 of 4 key points -> about 3/4 of the marks). NEVER award marks for a key point that "
            f"is present in the REFERENCE ANSWER but absent from the STUDENT ANSWER. If the STUDENT ANSWER "
            f"block is empty or has no real content addressing the question, the score is 0. Treat misspelled "
            f"or OCR-garbled forms of the correct term as correct, and IGNORE stray words or text that clearly "
            f"belongs to another question. Do NOT give marks for merely repeating the question wording.\n"
            f'Reply with compact JSON only: {{"score": <0..{maxm}>, "reason": "<=12 words, concrete>"}}.')

# ---- (1) empty-guarded LLM grade -------------------------------------------
def llm_grade(model_ans, student, maxm, question=""):
    if not globals().get("LLM_ENABLED", True):
        return None
    if student_is_empty(student):                       # HARD GUARD — never grade a blank
        return 0.0, "student left this part blank — 0"
    _k = _grade_key((model_ans or "").strip(), (student or "").strip(), float(maxm), question or "")
    if _k in _GRADE_CACHE:
        return _GRADE_CACHE[_k]
    try:
        txt = _llm_chat(_LLM_SYS, _build_grade_prompt(model_ans, student, maxm, question),
                        max_new_tokens=128)
        return _parse_grade(txt, maxm)
    except Exception:
        import traceback; traceback.print_exc()
        return None

# ---- (1) empty-guarded batch prime -----------------------------------------
def prime_llm_grades(qids):
    if not globals().get("LLM_ENABLED", True):
        return
    items, n_blank = [], 0
    for qid in qids:
        if (AUTO_RUBRIC.get(qid, {}) or {}).get("qtype") == "objective":
            continue
        student   = (answer_map.get(qid, "") or "").strip()
        model_ans = (MODEL_ANSWERS.get(qid, "") or "").strip()
        maxm      = float(QUESTION_MARKS[qid])
        qtext     = question_for(qid) if "question_for" in globals() else ""
        if student_is_empty(student):                   # blank -> auto 0, never sent to the LLM
            _GRADE_CACHE[_grade_key(model_ans, "", maxm, qtext)] = (0.0, "student left this part blank — 0")
            n_blank += 1
            continue
        items.append((model_ans, student, maxm, qtext))
    if n_blank:
        print(f"  [LLM] {n_blank} blank part(s) -> auto 0 (no model-answer leakage)")
    if not items:
        return
    import time
    t0 = time.time()
    print(f"  [LLM] batch-grading {len(items)} non-blank parts "
          f"(batch_size={globals().get('LLM_BATCH_SIZE',4)})…")
    texts = _llm_chat_batch([_LLM_SYS]*len(items),
                            [_build_grade_prompt(*it) for it in items],
                            max_new_tokens=int(globals().get("LLM_GRADE_TOKENS", 64)))
    for (ma, st, mm, qt), text in zip(items, texts):
        _GRADE_CACHE[_grade_key(ma, st, mm, qt)] = _parse_grade(text, mm)
    retry = [it for it in items if _GRADE_CACHE.get(_grade_key(*it)) is None]
    if retry:
        print(f"  [LLM] retrying {len(retry)} unparsed part(s) individually…")
        for (ma, st, mm, qt) in retry:
            try:
                txt = _llm_chat(_LLM_SYS, _build_grade_prompt(ma, st, mm, qt), max_new_tokens=128)
                _GRADE_CACHE[_grade_key(ma, st, mm, qt)] = _parse_grade(txt, mm)
            except Exception:
                pass
    print(f"  [LLM] grading finished in {time.time()-t0:.0f}s")

# ---- (3) garbled part-marker recovery --------------------------------------
# "02-(1)", "02.(1)", "Q2-(1)" : question number is fine, the (a) was OCR'd as a digit.
_QPART_GARBLED_RE = _re89.compile(
    rf"^\s*\[?\s*(?:{_QLAB}\s*[-.:]?\s*(?:no\.?)?\s*)?"
    rf"\(?\s*({_NUMC}{{1,2}})\s*[\)\].:\-]?\s*"
    rf"\(\s*([0-9a-hA-H])\s*\)"
    rf"[\):.\-]*(?:\s+|\s*$)", _re89.I)
def _detect_partmark(ln):
    m = _QPART_RE.match(ln)
    if m:
        num = _qnum_from(m.group(1)) if m.group(1) else None
        return num, m.group(2).lower(), m.end()
    g = _QPART_GARBLED_RE.match(ln)
    if g:
        num = _qnum_from(g.group(1))
        if num is not None:
            inner  = g.group(2).lower()
            letter = inner if inner in "abcdefgh" else None   # digit -> part unknown (positional)
            return num, letter, g.end()
    return None

# ---- (4) duplicate-marker routing: suppress the wrong number's content bonus ----
def _assign_by_marker(blocks, qorder, qp_parts, use_model):
    assign = [None] * len(blocks)
    seen, garbled, prev_q = set(), [], None
    for i, b in enumerate(blocks):
        n = b["num"]; q = f"Q{n}" if n is not None else None
        if q in qp_parts and q not in seen:
            assign[i] = q; seen.add(q); prev_q = q          # first sighting -> trust the marker
        elif n is None and prev_q is not None:
            assign[i] = prev_q                              # numberless -> continue previous
        else:
            garbled.append(i)                               # invalid OR duplicate number -> content
    if garbled:
        # NB: clear num on the copies so a misread duplicate (e.g. Q4a read as 'Q1a') does NOT
        # get its wrong number's +bonus and snap back onto the duplicated question.
        sub_blocks = [dict(blocks[i], num=None) for i in garbled]
        Ssub = _block_question_scores(sub_blocks, qorder, qp_parts, use_model)
        sub  = _assign_blocks(Ssub, qorder)
        for idx, q in zip(garbled, sub):
            assign[idx] = q
    by_mark = len(blocks) - len(garbled)
    return assign, f"marker ({by_mark}/{len(blocks)} by number, {len(garbled)} by content/dup)"

# ---- (5) embed every student diagram crop into the DOCX report -------------
def embed_student_diagrams(doc, html_path, valid_qids):
    import os, tempfile
    from docx.shared import Inches
    if not html_path or not os.path.exists(html_path):
        print("  [report] no student OCR HTML found -> diagrams not embedded"); return 0
    html = open(html_path, encoding="utf-8", errors="ignore").read()
    figs = _re89.findall(r"<figure\b.*?</figure>", html, _re89.S)
    if not figs:
        return 0
    _DG = str.maketrans({"S":"5","s":"5","l":"1","I":"1","O":"0","o":"0","B":"8","Z":"2","z":"2","g":"9"})
    _capq = _re89.compile(r"^\s*\(?\s*(?:q(?:ues(?:tion)?)?|ans(?:wer)?)?\s*[-.:]?\s*"
                          r"\(?\s*([0-9SslIOoBZzg]{1,2})\s*[-.\)\]\.: ]", _re89.I)
    _kw2q = {"chock":"Q6","bedplate":"Q6","tank top":"Q6","tanktop":"Q6",
             "tappet":"Q9","rocker arm":"Q9","valve spindle":"Q9","valve train":"Q9",
             "crankcase":"Q4","flame trap":"Q4","relief device":"Q4","deflector":"Q4"}
    def _qid_for(cap):
        m = _capq.match(cap or "")
        if m:
            d = _re89.search(r"\d{1,2}", m.group(1).translate(_DG))
            if d:
                q = f"Q{int(d.group(0))}"
                if q in valid_qids: return q, "marker"
        c = (cap or "").lower(); votes = {}
        for kw, q in _kw2q.items():
            if q in valid_qids and kw in c: votes[q] = votes.get(q, 0) + 1
        if votes:
            best = max(votes, key=votes.get)
            if list(votes.values()).count(votes[best]) == 1: return best, "content"
        return None, ""
    doc.add_heading("Student diagrams (extracted from the OCR)", 1)
    doc.add_paragraph("Every sketch Chandra OCR detected on the student sheet, matched to its question by "
                      "the written marker (else by content). Diagrams whose marker was OCR-garbled still "
                      "appear here so a human reviewer can see what the student actually drew.")
    n = 0
    for f in figs:
        b = _re89.search(r"data:image/(\w+);base64,([A-Za-z0-9+/=]+)", f)
        if not b: continue
        cap = _re89.search(r"<figcaption[^>]*>(.*?)</figcaption>", f, _re89.S)
        capT = _re89.sub(r"\s+", " ", _re89.sub(r"<[^>]+>", " ", cap.group(1))).strip() if cap else ""
        q, how = _qid_for(capT)
        try:
            raw = _b64_89.b64decode(b.group(2))
            tf = tempfile.NamedTemporaryFile(delete=False, suffix="." + (b.group(1) or "png"))
            tf.write(raw); tf.close()
        except Exception:
            continue
        n += 1
        head = doc.add_paragraph()
        r = head.add_run(f"Diagram {n} — " +
                         (f"{q}" + (f"  (matched by {how})" if how else "") if q
                          else "question unclear (OCR marker lost)"))
        r.bold = True
        try:
            doc.add_picture(tf.name, width=Inches(5.3))
        except Exception:
            pass
        if capT:
            cp = doc.add_paragraph(); cr = cp.add_run("OCR caption: " + capT[:300]); cr.italic = True
    print(f"  [report] embedded {n} student diagram(s)")
    return n

print("v8.9 fixes loaded: empty-guard + prompt-hardening + garbled-marker recovery "
      "+ duplicate routing + diagram embedding.")


## PART 2 — Question paper + answer key (run **once per paper**)

Two uploads: the **question paper** (marks per part) and the **correct answer key** (model answers).
The auto-rubric splits each part's model answer into key points for partial credit, but the
**mark totals come from the question paper**, not from auto-detection.

### 2.1 Upload the QUESTION PAPER → extract per-part marks + clean question text
If the paper is a normal (printed) PDF it has a text layer, so marks come straight from the
positioned `[N]` brackets — far more reliable than OCR. The clean per-question text is also
captured here and used later to **content-anchor** the key & student answers (robust to
OCR-garbled handwritten question numbers).

In [ ]:
def _upload(prompt):
    try:
        from google.colab import files as _f
        print(prompt); up = _f.upload(); return list(up.keys())[0]
    except Exception:
        return input(prompt + "\n(path) ").strip().strip('"').strip("'")

# ---- read the question paper's text layer with word positions (printed PDF) ----
def _qp_rows_from_pdf(path, ytol=4.0):
    import fitz
    rows = []
    d = fitz.open(path)
    for p in d:
        words = p.get_text("words")            # x0,y0,x1,y1,word,...
        if not words: continue
        words.sort(key=lambda w: (round(w[1]/ytol), w[0]))
        cur, buf = None, []
        def flush():
            if buf:
                buf.sort(key=lambda x: x[0])
                rows.append({"y": p.number*100000 + buf[0][1],
                             "text": " ".join(x[4] for x in buf)})
        for w in words:
            if cur is None or abs(w[1]-cur) <= ytol: buf.append(w); cur = w[1] if cur is None else cur
            else: flush(); buf = [w]; cur = w[1]
        flush()
    d.close()
    return rows

# number + SPACE -> a question header ("1 (a)", "9 With"); excludes note lines "1)"
_QP_HEAD = re.compile(r"^\s*(\d{1,2})(?=\s)")
def _qp_marks_and_text(rows):
    rubric, paper, cur, last = {}, {}, None, {}
    part_txt, stem = {}, {}            # per-part question text + shared question stem
    cur_part = None
    for r in sorted(rows, key=lambda r: r["y"]):
        t = r["text"]
        mh = _QP_HEAD.match(t)
        if mh and 1 <= int(mh.group(1)) <= 30:
            cur = f"Q{int(mh.group(1))}"; rubric.setdefault(cur, {}); paper.setdefault(cur, [])
            cur_part = None; stem.setdefault(cur, [])
        # marks
        mm = _MARKS_RE.search(t); pm = _PART_RE.search(t)
        if pm and cur:
            cur_part = pm.group(1).lower(); last[cur] = cur_part
            part_txt.setdefault(f"{cur}{cur_part}", [])
        if mm and cur:
            part = pm.group(1).lower() if pm else last.get(cur)
            if part: rubric[cur][part] = int(mm.group(1)); last[cur] = part
        # clean question text (strip [N] and the leading "N (a)"/"(a)" marker; drop footers)
        tt = _MARKS_RE.sub("", t).strip()
        if cur and tt and not tt.lower().startswith("page "):
            paper[cur].append(tt)
            body = re.sub(r"^\s*\d{0,2}\s*\(?[a-eA-E]?\)?\s*", "", tt) if (pm or mh) else tt
            if cur_part is not None:
                part_txt[f"{cur}{cur_part}"].append(body)
            else:
                stem[cur].append(tt)            # text before any part marker = shared stem
    paper = {q: " ".join(v) for q, v in paper.items()}
    # prepend each question's shared stem to every one of its parts
    PART = {}
    for qid, lines in part_txt.items():
        q = re.match(r"(Q\d+)", qid).group(1)
        s = " ".join(stem.get(q, [])).strip()
        PART[qid] = ((s + " ") if s else "") + " ".join(lines).strip()
    return rubric, paper, PART

QP_PATH = _upload("Upload the QUESTION PAPER (pdf / image):")
print("Question paper:", QP_PATH)

import os
qp_marks, PAPER_Q_TEXT, PAPER_PART_TEXT = {}, {}, {}
if QP_PATH.lower().endswith(".pdf"):
    try:
        _rows = _qp_rows_from_pdf(QP_PATH)
        if _rows:
            qp_marks, PAPER_Q_TEXT, PAPER_PART_TEXT = _qp_marks_and_text(_rows)
            print(f"Read marks + question text from the PDF text layer ({len(_rows)} lines).")
    except Exception as e:
        print("PDF text-layer read failed, will OCR:", e)

if not qp_marks:                       # image-only / scanned paper -> OCR fallback
    print("No text layer -> OCR'ing the question paper with Chandra ...")
    ensure_chandra(); free_gpu_for_ocr()
    _qp_imgs = [_cap_image(Image.fromarray(preprocess(np.array(im))["denoised"]).convert("RGB"), MAX_OCR_SIDE)
                for im in load_pages(QP_PATH)]
    _qp_raws = ocr_images_raw(_qp_imgs)
    _qp_rows = rows_from_raws(_qp_raws, _qp_imgs)
    qp_marks, PAPER_Q_TEXT, PAPER_PART_TEXT = _qp_marks_and_text(_qp_rows)
    if not qp_marks:
        qp_marks = extract_qp_marks_from_text("\n".join(parse_markdown(r) for r in _qp_raws))

def question_for(qid):
    """The exam question text for a (question-)part qid, for the grader & LLM."""
    if qid in PAPER_PART_TEXT and PAPER_PART_TEXT[qid].strip():
        return PAPER_PART_TEXT[qid]
    base = re.match(r"(Q\d+)", qid).group(1)
    return PAPER_Q_TEXT.get(base, "")

QP_MARKS, QP_PARTS = qp_to_qids(qp_marks)
print("\nExtracted marks per part (EDIT QP_MARKS in 2.2 if any are wrong):")
for q in sorted(QP_PARTS, key=lambda x:int(x[1:])):
    if QP_PARTS[q] == ["_"]:
        print(f"  {q}: {QP_MARKS[q]}")
    else:
        print(f"  {q}: " + ", ".join(f"{p}={QP_MARKS[q+p]}" for p in QP_PARTS[q])
              + f"  (= {sum(QP_MARKS[q+p] for p in QP_PARTS[q]):g})")
print(f"\nTotal questions: {len(QP_PARTS)} | total marks on paper: {sum(QP_MARKS.values()):g}")


### 2.2 (Optional) fix any mis-read marks
```python
# QP_MARKS.update({'Q8b': 4, 'Q8c': 4})
# QP_PARTS['Q8'] = ['a','b','c']
```

In [ ]:
# QP_MARKS.update({'Q1a':2,'Q1b':2,'Q1c':2',Q1d':2',Q1e':2,'Q2':7,'Q3':10,'Q4':5,'Q5Ii':2,'Q5IIi':2})   # e.g. {'Q8b':4,'Q8c':4}
# QP_PARTS['Q8']=['a','b','c']
print('QP marks locked:', {k:QP_MARKS[k] for k in sorted(QP_MARKS)})

### 2.3 Upload the ANSWER KEY → OCR → split per part → build the key + **download key HTML**

In [ ]:
KEY_PATH = _upload("Upload the CORRECT ANSWER KEY (pdf / image / docx / txt):")
print("Answer key:", KEY_PATH)
import os
_ext = os.path.splitext(KEY_PATH)[1].lower()

ensure_chandra()
KEY_HTML_PATH = None
if _ext in (".pdf",".png",".jpg",".jpeg",".webp",".tif",".tiff",".bmp"):
    free_gpu_for_ocr()
    _key_imgs_full, _key_idx = select_pages(KEY_PATH, skip_leading=KEY_SKIP_LEADING, skip_blanks=KEY_SKIP_BLANKS)
    _key_prepped = [_cap_image(Image.fromarray(preprocess(np.array(im))["denoised"]).convert("RGB"), MAX_OCR_SIDE)
                    for im in _key_imgs_full]
    _key_raws = ocr_images_raw(_key_prepped)
    key_lines = lines_from_ocr(_key_raws, _key_prepped)   # v8.3 diagram-aware source
    # downloadable structured HTML for the key
    KEY_HTML_PATH = "answer_key_ocr.html"
    structured_html_from_raws(_key_raws, _key_prepped, _key_idx, KEY_HTML_PATH,
                              "Answer key — " + os.path.basename(KEY_PATH))
else:
    key_lines = lines_from_ocr_pages(read_any(KEY_PATH))

unload_chandra()                                # free Chandra BEFORE the BGE content-anchor
grading_models_to_gpu()
# Content-anchored: each key segment is mapped to its paper question by CONTENT (robust to
# the key's OCR-garbled handwritten numbers like Q3->'A18', Q4->'Q1', Q9->'Q.'/'89.').
globals()["_N_QUESTIONS"] = max([int(q[1:]) for q in QP_PARTS], default=200); globals()["_QTYPE"] = {}
key_part_map = map_parts_anchored(key_lines, QP_PARTS, PAPER_Q_TEXT, label="key")
MODEL_ANSWERS, QUESTION_MARKS, AUTO_RUBRIC, MCQ_OPTION, KEY_PARTS = {}, {}, {}, {}, {}
for qid, mk in QP_MARKS.items():
    ans = key_part_map.get(qid, "")
    MODEL_ANSWERS[qid]  = ans
    QUESTION_MARKS[qid] = float(mk)
    opt = detect_mcq_option(ans)
    qtype, pts = build_auto_rubric(ans, float(mk))
    if opt: qtype = "objective"
    AUTO_RUBRIC[qid] = {"qtype": qtype, "points": pts, "option": opt}
    MCQ_OPTION[qid]  = opt
    globals()["_QTYPE"][int(re.match(r"Q(\d+)", qid).group(1))] = qtype
# per-question part map (model answers) for aligning the student's parts later
for q, parts in QP_PARTS.items():
    if parts != ["_"]:
        KEY_PARTS[q] = OrderedDict((p, MODEL_ANSWERS.get(f"{q}{p}", "")) for p in parts)

QIDS = sorted(QP_MARKS, key=lambda x: (int(re.match(r"Q(\d+)", x).group(1)), x))
TOTAL_MAX = round(sum(QUESTION_MARKS.values()), 2)
_missing = [qid for qid in QIDS if not MODEL_ANSWERS[qid].strip()]
print(f"\nKey built: {len(QIDS)} gradable parts | total = {TOTAL_MAX} marks")
if _missing: print(f"  [warn] no key text parsed for: {_missing} (check the key OCR / markers)")
for qid in QIDS:
    e = AUTO_RUBRIC[qid]
    print(f"  {qid}: {QUESTION_MARKS[qid]:>4} mk ({e['qtype']}) — {MODEL_ANSWERS[qid][:60]}")
print("\nKey + rubric locked in memory. Run PART 3 per student.")


### 2.4 (Re-runnable) Download the answer-key OCR HTML
Run this any time to re-download `answer_key_ocr.html` — no need to re-run the OCR.

In [ ]:
import os
if KEY_HTML_PATH and os.path.exists(KEY_HTML_PATH):
    print("Key OCR HTML:", os.path.abspath(KEY_HTML_PATH), f"({os.path.getsize(KEY_HTML_PATH)/1e6:.2f} MB)")
    try:
        from google.colab import files as _f; _f.download(KEY_HTML_PATH)
    except Exception:
        print("(not on Colab — open the path above)")
else:
    print("No key HTML yet — run 2.3 first.")


## PART 3 — Student answer sheet (**re-run per student**)
Upload one sheet → leading pages + blanks skipped → OCR → HTML → parts mapped to the key → graded → DOCX.

### 3.1 Upload ONE student sheet → skip leading/blank pages → OCR → **download student HTML**

In [ ]:
STUDENT_PATH = _upload("Upload the STUDENT ANSWER SHEET (pdf / image):")
SOURCE_LABEL = os.path.basename(STUDENT_PATH)
print("Student file:", SOURCE_LABEL)

ensure_chandra(); free_gpu_for_ocr()
_stu_imgs_full, _stu_idx = select_pages(STUDENT_PATH, skip_leading=SKIP_LEADING_PAGES, skip_blanks=True)
_stu_prepped = [_cap_image(Image.fromarray(preprocess(np.array(im))["denoised"]).convert("RGB"), MAX_OCR_SIDE)
                for im in _stu_imgs_full]
_stu_raws = ocr_images_raw(_stu_prepped)
student_lines = lines_from_ocr(_stu_raws, _stu_prepped)   # v8.3 diagram-aware source
STUDENT_HTML_PATH = f"{os.path.splitext(SOURCE_LABEL)[0]}_ocr.html"
structured_html_from_raws(_stu_raws, _stu_prepped, _stu_idx, STUDENT_HTML_PATH,
                          "Student — " + SOURCE_LABEL)
unload_chandra()
print(f"\nBuilt {len(student_lines)} student lines from {len(_stu_idx)} page(s).")
print("\n".join(student_lines[:12]))


### 3.1b (Re-runnable) Download the student OCR HTML
Run this any time to re-download the student's `..._ocr.html` — no need to re-run the OCR.

In [ ]:
import os
if "STUDENT_HTML_PATH" in globals() and os.path.exists(STUDENT_HTML_PATH):
    print("Student OCR HTML:", os.path.abspath(STUDENT_HTML_PATH), f"({os.path.getsize(STUDENT_HTML_PATH)/1e6:.2f} MB)")
    try:
        from google.colab import files as _f; _f.download(STUDENT_HTML_PATH)
    except Exception:
        print("(not on Colab — open the path above)")
else:
    print("No student HTML yet — run 3.1 first.")


### 3.2 Map answers to questions → **split into parts** → align to the key's parts

In [ ]:
grading_models_to_gpu()
# Content-anchored: map each student segment to its paper question by CONTENT, robust to
# however the student numbered things. Falls back to label-based map_parts if no paper text.
answer_map = map_parts_anchored(student_lines, QP_PARTS, PAPER_Q_TEXT, label="student")
for qid in QIDS: answer_map.setdefault(qid, "")
_have = sum(1 for qid in QIDS if answer_map[qid].strip())
print(f"Mapped: {_have}/{len(QIDS)} parts have a student answer.")
for qid in QIDS:
    tag = "ok " if answer_map[qid].strip() else "-- "
    print(f"  {tag}{qid:<6} {answer_map[qid][:70]}")


### 3.3 Grade every part (Qwen leads · STS/NLI flags)

In [ ]:
import pandas as pd
def _mk(x): return "-" if x is None else x
graded, rows = {}, []
prime_llm_grades(QIDS)          # v8.5: one batched LLM pass (fast) -> per-part calls are cached
for qid in QIDS:
    d = grade_question(qid); graded[qid] = d
    flag = ("REVIEW" if d.get("review") else "PERFECT" if d["perfect"]
            else "CONTRADICTION" if d["contradiction"] else "OFF-TOPIC" if d["off_topic"] else "")
    rows.append({"Part": qid, "Max": d["max_marks"],
                 "LLM": _mk(d.get("llm_marks")), "Engine": _mk(d.get("engine_marks")),
                 "Final": d["marks"], "Flag": flag, "Decision": d.get("decision",""),
                 "Why": (d["llm_reason"] or d["why"])[:80]})
results = pd.DataFrame(rows)
TOTAL = round(results["Final"].sum(), 2)
print("="*82); print(f"  EVALUATION — {SOURCE_LABEL}   (LLM leads · STS/NLI engine cross-checks)"); print("="*82)
print(f"  {'Part':<6}{'LLM':>6}{'Engine':>8}{'Final':>7}{'Max':>5}  {'Flag':<14}Reason")
print("-"*82)
for _, r in results.iterrows():
    print(f"  {r['Part']:<6}{str(r['LLM']):>6}{str(r['Engine']):>8}{r['Final']:>7}{r['Max']:>5}  "
          f"{('<'+r['Flag']+'>') if r['Flag'] else '':<14}{r['Why']}")
print("-"*82)
_rev = [q for q in QIDS if graded[q].get("review")]
print(f"  TOTAL = {TOTAL} / {TOTAL_MAX}" + (f"   |  flagged for review: {', '.join(_rev)}" if _rev else ""))
print("="*82)
results


### 3.4 Export the DOCX report (per-part marks, flags, LLM vs engine, HTML links) + download

In [ ]:
from docx import Document
from docx.shared import Pt
OUT_DOCX = f"evaluation_{os.path.splitext(SOURCE_LABEL)[0]}.docx"
doc = Document()
doc.add_heading("Answer-Sheet Evaluation Report", 0)
doc.add_paragraph(f"Student sheet : {SOURCE_LABEL}")
doc.add_paragraph(f"Question paper: {os.path.basename(QP_PATH)}   |   Answer key: {os.path.basename(KEY_PATH)}")
doc.add_paragraph(f"Grader: {LLM_MODEL_ID} (leads) + {STS_MODEL_ID}/{NLI_MODEL_ID} (flagging) | "
                  f"blend {W_RUBRIC}/{W_SEM} | marks rounded to {ROUND_STEP}")
doc.add_paragraph("OCR HTML: answer_key_ocr.html , "
                  f"{os.path.splitext(SOURCE_LABEL)[0]}_ocr.html (equations + diagrams, offline)")
doc.add_paragraph("Generated by the AI evaluation pipeline of Krishnarsh & Nancy (C-DAC Noida).")

t = doc.add_table(rows=1, cols=7); t.style = "Light Grid Accent 1"
for j, h in enumerate(["Part","LLM","Engine","Final","Max","Flag","How decided"]):
    t.rows[0].cells[j].text = h
for qid in QIDS:
    d = graded[qid]; c = t.add_row().cells
    c[0].text = qid
    c[1].text = "-" if d.get("llm_marks") is None else f"{d['llm_marks']}"
    c[2].text = "-" if d.get("engine_marks") is None else f"{d['engine_marks']}"
    c[3].text = f"{d['marks']}"
    c[4].text = f"{d['max_marks']:g}"
    c[5].text = ("REVIEW" if d.get("review") else "PERFECT" if d["perfect"]
                 else "CONTRA" if d["contradiction"] else "OFF-TOPIC" if d["off_topic"] else "-")
    c[6].text = d.get("decision","")
doc.add_heading("Summary", 1)
doc.add_paragraph(f"TOTAL = {TOTAL} / {TOTAL_MAX}  ({round(100*TOTAL/max(TOTAL_MAX,1e-9),1)}%)")
doc.add_paragraph(f"Columns: LLM = {LLM_MODEL_ID} marks — the LLM is the SOLE grader. "
                  "Engine = STS+NLI marks, shown for reference only; the engine never changes the score. "
                  "Final = the LLM's marks. When the engine disagrees with the LLM by more than "
                  f"{LLM_DISAGREE:g} (fraction) the part is flagged REVIEW for a human to check.")
_flagged = [qid for qid in QIDS if graded[qid].get("review")]
if _flagged:
    doc.add_paragraph("Flagged for manual review (LLM vs engine disagreed): " + ", ".join(_flagged))

doc.add_heading("Per-part detail", 1)
for qid in QIDS:
    d = graded[qid]
    doc.add_heading(f"{qid} — {d['marks']}/{d['max_marks']:g}" + ("   [REVIEW]" if d.get("review") else ""), 2)
    if d.get("question"): doc.add_paragraph(f"Question: {d['question'][:400]}")
    doc.add_paragraph(f"Model answer: {MODEL_ANSWERS.get(qid,'')[:400]}")
    doc.add_paragraph(f"Student answer: {(answer_map.get(qid,'') or '(empty)')[:400]}")
    _llm = "-" if d.get("llm_marks") is None else f"{d['llm_marks']}/{d['max_marks']:g}"
    _eng = "-" if d.get("engine_marks") is None else f"{d['engine_marks']}/{d['max_marks']:g}"
    p = doc.add_paragraph()
    p.add_run("LLM (Qwen): ").bold = True
    p.add_run(f"{_llm}" + (f" — {d['llm_reason']}" if d.get("llm_reason") else ""))
    p2 = doc.add_paragraph()
    p2.add_run("Engine (STS+NLI): ").bold = True
    p2.add_run(f"{_eng}")
    p3 = doc.add_paragraph()
    p3.add_run("Decision: ").bold = True
    p3.add_run(f"{d.get('decision','')}  ->  Final {d['marks']}/{d['max_marks']:g}")
    if d["qtype"] != "objective" and d["points"]:
        pt = doc.add_table(rows=1, cols=4); pt.style = "Light List Accent 1"
        for j, h in enumerate(["Key point (from model answer)","Marks","How","Awarded"]):
            pt.rows[0].cells[j].text = h
        for pp in d["points"]:
            c = pt.add_row().cells
            c[0].text=pp["text"][:120]; c[1].text=str(pp["marks"]); c[2].text=pp["how"]; c[3].text=str(pp["awarded"])
# v8.9: embed every student diagram crop pulled from the OCR HTML
import re as _re_diag
_valid_q = {_re_diag.match(r"(Q\d+)", q).group(1) for q in QIDS if _re_diag.match(r"(Q\d+)", q)}
try:
    embed_student_diagrams(doc, globals().get("STUDENT_HTML_PATH"), _valid_q)
except Exception as _e:
    print("  [report] diagram embed skipped:", _e)
doc.save(OUT_DOCX); print("Saved:", OUT_DOCX)
try:
    from google.colab import files as _f
    for fn in [KEY_HTML_PATH, STUDENT_HTML_PATH, OUT_DOCX]:
        if fn and os.path.exists(fn): _f.download(fn)
except Exception:
    print("Download manually:", os.path.abspath(OUT_DOCX))


---
**Next student:** re-run Part 3 only. The question paper, key and rubric from Part 2 stay in memory.